In [3]:
pip install datasets

     ---------------------------------------- 0.0/107.2 kB ? eta -:--:--
     -------------------------------------- 107.2/107.2 kB 6.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   --------------------------------------- 559.1/559.1 kB 36.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/120.0 kB ? eta -:--:--
   ---------------------------------------- 120.0/120.0 kB 6.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/203.9 kB ? eta -:--:--
   ---------------------------------------- 203.9/203.9 kB ? eta 0:00:00
   ---------------------------------------- 0.0/795.8 kB ? eta -:--:--
   --------------------------------------- 795.8/795.8 kB 25.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/99.9 kB ? eta -:--:--
   ---------------------------------------- 99.9/99.9 kB 5.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/150.3 kB ? eta -:--:--
   ----------------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\masta\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [4]:
from datasets import load_dataset
import pandas as pd
import numpy as np

ds = load_dataset(
    "arubique/disco-model-outputs",
    "mmlu_abstract_algebra",
    split="train",
)

models = load_dataset(
    "arubique/disco-model-outputs",
    "models",
    split="train",
)

df = ds.to_pandas()
models_df = models.to_pandas()

print(models_df.head())
print(df.head())

README.md:   0%|          | 0.00/38.7k [00:00<?, ?B/s]

C:\Users\masta\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\masta\.cache\huggingface\hub\datasets--arubique--disco-model-outputs. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


mmlu_abstract_algebra/train-00000-of-000(…): reconstructing file:   0%|          |  0.00B / 1.41MB            

mmlu_abstract_algebra/train-00000-of-000(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/42500 [00:00<?, ? examples/s]

models/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.1kB            

models/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/425 [00:00<?, ? examples/s]

   model_idx                                         model_name
0          0  open-llm-leaderboard/details_abacusai__MetaMat...
1          1  open-llm-leaderboard/details_zhengr__MixTAO-7B...
2          2  open-llm-leaderboard/details_alignment-handboo...
3          3  open-llm-leaderboard/details_LoSboccacc__ortho...
4          4  open-llm-leaderboard/details_rombodawg__Leader...
   sample_idx  model_idx  correctness   logit_0   logit_1   logit_2   logit_3
0           0          0          0.0 -1.430295 -4.180295 -1.305295 -0.805295
1           0          1          1.0 -9.769019 -0.722144 -2.253394 -0.894019
2           0          2          1.0 -1.161713 -0.911713 -1.536713 -2.661713
3           0          3          0.0 -2.395193 -1.504568 -0.660818 -1.770193
4           0          4          0.0 -1.182963 -1.307963 -2.182963 -1.182963


In [5]:
x = (
    df[df["model_idx"] == 0]
    .sort_values("sample_idx")
    .head(5)
    .copy()
)

L = x[
    ["logit_0", "logit_1", "logit_2", "logit_3"]
].to_numpy()

Z = L - L.max(axis=1, keepdims=True)

P = np.exp(Z)
P = P / P.sum(axis=1, keepdims=True)

x["pred_idx"] = L.argmax(axis=1)

for j, c in enumerate("ABCD"):
    x[f"P_{c}"] = P[:, j]

print(x)

      sample_idx  model_idx  correctness   logit_0   logit_1   logit_2  \
0              0          0          0.0 -1.430295 -4.180295 -1.305295   
425            1          0          0.0 -2.836474 -0.711474 -1.086474   
850            2          0          1.0 -2.562807 -0.562807 -3.687807   
1275           3          0          1.0 -2.756954 -3.506954 -1.381954   
1700           4          0          0.0 -0.704608 -5.079608 -0.829608   

       logit_3  pred_idx       P_A       P_B       P_C       P_D  
0    -0.805295         3  0.245983  0.015725  0.278735  0.459557  
425  -2.836474         1  0.062006  0.519169  0.356819  0.062006  
850  -5.125307         1  0.113755  0.840542  0.036931  0.008772  
1275 -2.006954         2  0.132548  0.062611  0.524237  0.280604  
1700 -2.829608         0  0.496397  0.006249  0.438069  0.059286  


In [6]:
pip install huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\masta\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [9]:
from huggingface_hub import hf_hub_url
import requests
import pandas as pd

repo_id = (
    "open-llm-leaderboard-old/"
    "details_Ba2han__Phi-3-Medium-Llamaish"
)

filename = (
    "2024-06-03T12-48-11.919525/"
    "details_harness|hendrycksTest-abstract_algebra|5_"
    "2024-06-03T12-48-11.919525.parquet"
)

# Hugging Face上の実ファイルURLを生成
url = hf_hub_url(
    repo_id=repo_id,
    filename=filename,
    repo_type="dataset"
)

print(url)

# ダウンロード
r = requests.get(url, stream=True)
r.raise_for_status()

# Windowsで使える安全な名前にして保存
local_path = "abstract_algebra.parquet"

with open(local_path, "wb") as f:
    for chunk in r.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print("Downloaded:", local_path)

# Parquetを読む
df = pd.read_parquet(local_path)

print(df.columns)
print(df.head())

https://huggingface.co/datasets/open-llm-leaderboard-old/details_Ba2han__Phi-3-Medium-Llamaish/resolve/main/2024-06-03T12-48-11.919525/details_harness%7ChendrycksTest-abstract_algebra%7C5_2024-06-03T12-48-11.919525.parquet
Downloaded: abstract_algebra.parquet
Index(['choices', 'cont_tokens', 'example', 'full_prompt', 'gold',
       'gold_index', 'input_tokens', 'instruction', 'metrics',
       'num_asked_few_shots', 'num_effective_few_shots', 'padded',
       'pred_logits', 'predictions', 'truncated'],
      dtype='object')
  choices                   cont_tokens  \
0      []  [[319], [350], [315], [360]]   
1      []  [[319], [350], [315], [360]]   
2      []  [[319], [350], [315], [360]]   
3      []  [[319], [350], [315], [360]]   
4      []  [[319], [350], [315], [360]]   

                                             example  \
0  Statement 1 | Some abelian group of order 45 h...   
1  Find the characteristic of the ring Z_3 x 3Z.\...   
2  Find all cosets of the subgroup 4Z of 2Z

In [10]:
print(df.shape)

print(df.columns.tolist())

print(df.head().to_string())

(100, 15)
['choices', 'cont_tokens', 'example', 'full_prompt', 'gold', 'gold_index', 'input_tokens', 'instruction', 'metrics', 'num_asked_few_shots', 'num_effective_few_shots', 'padded', 'pred_logits', 'predictions', 'truncated']
  choices                   cont_tokens                                                                                                                                                                                                                                                                                                   example                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [11]:
for i in range(5):
    print("----", i, "----")
    print("choices     :", df.iloc[i].get("choices"))
    print("gold        :", df.iloc[i].get("gold"))
    print("gold_index  :", df.iloc[i].get("gold_index"))
    print("predictions :", df.iloc[i].get("predictions"))
    print("metrics     :", df.iloc[i].get("metrics"))

---- 0 ----
choices     : []
gold        : []
gold_index  : []
predictions : [-3.09229207 -2.21729207 -3.59229207 -0.21729198]
metrics     : {'acc': 0.0, 'acc_norm': 0.0}
---- 1 ----
choices     : []
gold        : []
gold_index  : []
predictions : [-1.80456805 -0.30456799 -2.67956805 -3.80456805]
metrics     : {'acc': 0.0, 'acc_norm': 0.0}
---- 2 ----
choices     : []
gold        : []
gold_index  : []
predictions : [-2.61607051 -0.61607051 -1.24107051 -2.36607051]
metrics     : {'acc': 1.0, 'acc_norm': 1.0}
---- 3 ----
choices     : []
gold        : []
gold_index  : []
predictions : [-2.33315611 -1.70815623 -0.83315623 -1.33315623]
metrics     : {'acc': 1.0, 'acc_norm': 1.0}
---- 4 ----
choices     : []
gold        : []
gold_index  : []
predictions : [-3.31778169 -4.56778145 -0.06778169 -4.69278145]
metrics     : {'acc': 1.0, 'acc_norm': 1.0}


In [12]:
print(df.columns.tolist())

['choices', 'cont_tokens', 'example', 'full_prompt', 'gold', 'gold_index', 'input_tokens', 'instruction', 'metrics', 'num_asked_few_shots', 'num_effective_few_shots', 'padded', 'pred_logits', 'predictions', 'truncated']


In [13]:
cols = [
    "example",
    "full_prompt",
    "specifics",
    "choices",
    "gold",
    "gold_index",
    "predictions",
    "metrics"
]

for i in range(5):
    print("\n========================")
    print("ROW", i)
    print("========================")

    for col in cols:
        if col in df.columns:
            value = df.iloc[i][col]

            print(f"\n--- {col} ---")

            # 長すぎる場合は先頭2000文字
            s = str(value)

            if len(s) > 2000:
                print(s[:2000])
            else:
                print(s)


ROW 0

--- example ---
Statement 1 | Some abelian group of order 45 has a subgroup of order 10. Statement 2 | A subgroup H of a group G is a normal subgroup if and only if thenumber of left cosets of H is equal to the number of right cosets of H.
A. True, True
B. False, False
C. True, False
D. False, True
Answer:

--- full_prompt ---
The following are multiple choice questions (with answers) about abstract algebra.

Find all c in Z_3 such that Z_3[x]/(x^2 + c) is a field.
A. 0
B. 1
C. 2
D. 3
Answer: B

Statement 1 | If aH is an element of a factor group, then |aH| divides |a|. Statement 2 | If H and K are subgroups of G then HK is a subgroup of G.
A. True, True
B. False, False
C. True, False
D. False, True
Answer: B

Statement 1 | Every element of a group generates a cyclic subgroup of the group. Statement 2 | The symmetric group S_10 has 10 elements.
A. True, True
B. False, False
C. True, False
D. False, True
Answer: C

Statement 1| Every function from a finite set onto itself must b

In [14]:
from datasets import load_dataset
import numpy as np
import pandas as pd

# 元MMLU
mmlu = load_dataset(
    "cais/mmlu",
    "abstract_algebra",
    split="test"
)

# MMLU question -> gold index
question_to_gold = {
    item["question"].strip(): int(item["answer"])
    for item in mmlu
}

def find_gold(example):
    example = str(example)

    # 各MMLU questionがexample内に含まれるか確認
    hits = []

    for question, gold in question_to_gold.items():
        if question in example:
            hits.append((question, gold))

    if len(hits) == 1:
        return hits[0]

    return None, None


results = []

for i, row in df.iterrows():

    question, gold = find_gold(row["example"])

    ll = np.asarray(row["predictions"], dtype=float)

    # softmax
    z = ll - ll.max()
    probs = np.exp(z)
    probs /= probs.sum()

    pred = int(np.argmax(ll))

    saved_acc = float(row["metrics"]["acc"])

    if gold is not None:
        reconstructed_acc = int(pred == gold)
        p_correct = probs[gold]
    else:
        reconstructed_acc = np.nan
        p_correct = np.nan

    results.append({
        "row": i,
        "question": question,

        "gold_index": gold,
        "gold": "ABCD"[gold] if gold is not None else None,

        "loglik_A": ll[0],
        "loglik_B": ll[1],
        "loglik_C": ll[2],
        "loglik_D": ll[3],

        "P_A": probs[0],
        "P_B": probs[1],
        "P_C": probs[2],
        "P_D": probs[3],

        "pred": "ABCD"[pred],

        "p_correct": p_correct,

        "saved_acc": saved_acc,
        "reconstructed_acc": reconstructed_acc,
    })

result = pd.DataFrame(results)

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

C:\Users\masta\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\masta\.cache\huggingface\hub\datasets--cais--mmlu. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

abstract_algebra/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 9.96kB            

abstract_algebra/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

abstract_algebra/dev-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B / 3.45kB            

abstract_algebra/dev-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

In [15]:
matched = result["gold_index"].notna()

print("Gold matched:",
      matched.sum(), "/", len(result))

print("Accuracy agreement:",
      (
          result.loc[matched, "saved_acc"]
          ==
          result.loc[matched, "reconstructed_acc"]
      ).mean()
)

Gold matched: 100 / 100
Accuracy agreement: 1.0


In [16]:
print(
    result[
        [
            "row",
            "gold",
            "pred",
            "p_correct",
            "saved_acc",
            "reconstructed_acc"
        ]
    ].head(20)
)

    row gold pred  p_correct  saved_acc  reconstructed_acc
0     0    B    D   0.110390        0.0                  0
1     1    A    B   0.165731        0.0                  0
2     2    B    B   0.542190        1.0                  1
3     3    C    C   0.445132        1.0                  1
4     4    C    C   0.943675        1.0                  1
5     5    D    D   0.566656        1.0                  1
6     6    C    A   0.383271        0.0                  0
7     7    D    D   0.542897        1.0                  1
8     8    B    C   0.100307        0.0                  0
9     9    D    D   0.669433        1.0                  1
10   10    B    B   0.959664        1.0                  1
11   11    C    A   0.126552        0.0                  0
12   12    A    C   0.380001        0.0                  0
13   13    D    D   0.341824        1.0                  1
14   14    D    A   0.161793        0.0                  0
15   15    C    B   0.421790        0.0                 

In [19]:
from huggingface_hub import HfApi, hf_hub_url
from datasets import load_dataset

import pandas as pd
import numpy as np
import requests
import re

from io import BytesIO


# ============================================================
# 1. 設定
# ============================================================

MODEL = "Ba2han/Phi-3-Medium-Llamaish"

REPO_ID = (
    "open-llm-leaderboard-old/"
    "details_Ba2han__Phi-3-Medium-Llamaish"
)

# 今回実際に検証したrun
RUN = "2024-06-03T12-48-11.919525"

OUTPUT_CSV = "MMLU_Phi-3-Medium-Llamaish.csv"


# ============================================================
# 2. Hugging Face上のファイル一覧を取得
# ============================================================

api = HfApi()

files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)


# ============================================================
# 3. このrunのMMLU (hendrycksTest) Parquetだけ抽出
# ============================================================

mmlu_files = [
    f for f in files
    if (
        f.startswith(RUN + "/")
        and "hendrycksTest-" in f
        and f.endswith(".parquet")
    )
]

print("MMLU parquet files found:", len(mmlu_files))


# ============================================================
# 4. subject名を取り出す関数
#
# 例:
# details_harness|hendrycksTest-abstract_algebra|5_....
#
# -> abstract_algebra
# ============================================================

def extract_subject(filename):

    m = re.search(
        r"hendrycksTest-(.+?)\|(\d+)_",
        filename
    )

    if m is None:
        raise ValueError(
            f"Could not extract subject from: {filename}"
        )

    subject = m.group(1)
    num_fewshot_from_filename = int(m.group(2))

    return subject, num_fewshot_from_filename


# subject一覧を確認
subjects = []

for f in mmlu_files:
    subject, nshot = extract_subject(f)
    subjects.append(subject)

subjects = sorted(subjects)

print("Number of subjects:", len(subjects))
print(subjects)


# ============================================================
# 5. Hugging Face上のParquetを
#    Windowsに保存せず、直接メモリ上で読む
# ============================================================

def load_remote_parquet(filename):

    url = hf_hub_url(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset"
    )

    r = requests.get(
        url,
        timeout=120
    )

    r.raise_for_status()

    return pd.read_parquet(
        BytesIO(r.content)
    )


# ============================================================
# 6. stable softmax
#
# predictions =
# [loglik_A, loglik_B, loglik_C, loglik_D]
#
# から、A-D内で正規化した確率を作る
# ============================================================

def softmax4(loglik):

    x = np.asarray(
        loglik,
        dtype=float
    )

    x = x - np.max(x)

    p = np.exp(x)

    return p / np.sum(p)


# ============================================================
# 7. MMLU元データとの対応
#
# details側ではgold/gold_indexが空なので、
# exampleに含まれる問題文とcais/mmluを照合
# ============================================================

def normalize_text(s):
    """
    改行や連続スペースの違いを吸収して比較する
    """
    return re.sub(
        r"\s+",
        " ",
        str(s)
    ).strip()


def build_gold_lookup(subject):

    ds = load_dataset(
        "cais/mmlu",
        subject,
        split="test"
    )

    lookup = {}

    for item in ds:

        question = item["question"].strip()
        choices = item["choices"]
        gold_index = int(item["answer"])

        # Leaderboardのexampleと同じ形式を再構築
        formatted = (
            f"{question}\n"
            f"A. {choices[0]}\n"
            f"B. {choices[1]}\n"
            f"C. {choices[2]}\n"
            f"D. {choices[3]}\n"
            f"Answer:"
        )

        key = normalize_text(formatted)

        if key not in lookup:
            lookup[key] = []

        lookup[key].append({
            "question": question,
            "gold_index": gold_index,
            "choices": choices
        })

    return lookup


def find_gold(example, lookup):

    key = normalize_text(example)

    hits = lookup.get(key, [])

    if len(hits) == 1:

        return (
            hits[0]["question"],
            hits[0]["gold_index"]
        )

    elif len(hits) == 0:

        return None, None

    else:
        # 完全に同一のquestion+choicesが複数ある場合
        # goldが全部同じなら問題なし
        golds = {
            x["gold_index"]
            for x in hits
        }

        if len(golds) == 1:
            return (
                hits[0]["question"],
                hits[0]["gold_index"]
            )

        print(
            "WARNING: ambiguous exact match:",
            example
        )

        return None, None


# ============================================================
# 8. 全57分野を処理
# ============================================================

all_rows = []


for file_no, filename in enumerate(
    sorted(mmlu_files),
    start=1
):

    subject, filename_fewshot = extract_subject(
        filename
    )

    print(
        f"[{file_no}/{len(mmlu_files)}] "
        f"Processing: {subject}"
    )

    # ------------------------------------
    # Leaderboard details
    # ------------------------------------

    df = load_remote_parquet(
        filename
    )

    # ------------------------------------
    # 元MMLU
    # ------------------------------------

    gold_lookup = build_gold_lookup(
        subject
    )

    # ------------------------------------
    # 各問題
    # ------------------------------------

    for row_idx, row in df.iterrows():

        # ==========
        # Gold
        # ==========

        question, gold_index = find_gold(
            row["example"],
            gold_lookup
        )

        if gold_index is None:

            print(
                f"WARNING: gold not found "
                f"{subject}, row={row_idx}"
            )

            continue

        gold = "ABCD"[gold_index]


        # ==========
        # Log likelihood
        # ==========

        ll = np.asarray(
            row["predictions"],
            dtype=float
        )

        if len(ll) != 4:

            print(
                f"WARNING: predictions != 4 "
                f"{subject}, row={row_idx}"
            )

            continue


        loglik_A = ll[0]
        loglik_B = ll[1]
        loglik_C = ll[2]
        loglik_D = ll[3]


        # ==========
        # Softmax probability
        # ==========

        probs = softmax4(ll)

        p_A = probs[0]
        p_B = probs[1]
        p_C = probs[2]
        p_D = probs[3]


        # ==========
        # Prediction
        # ==========

        pred_index = int(
            np.argmax(ll)
        )

        pred = "ABCD"[pred_index]


        # ==========
        # Correct-answer probability
        # ==========

        p_correct = float(
            probs[gold_index]
        )


        # ==========
        # Accuracy
        # ==========

        accuracy = float(
            row["metrics"]["acc"]
        )


        # ==========
        # few-shot
        # ==========

        num_fewshot = row[
            "num_effective_few_shots"
        ]

        if pd.isna(num_fewshot):
            num_fewshot = filename_fewshot

        num_fewshot = int(num_fewshot)


        # ==========
        # temperature
        #
        # MMLUはlog-likelihood評価なので
        # sampling temperatureは非該当
        # ==========

        temperature = np.nan


        # ==========
        # 保存
        # ==========

        all_rows.append({

            "model":
                MODEL,

            "subject":
                subject,

            "question":
                question,

            "gold":
                gold,

            "pred":
                pred,

            "loglik_A":
                loglik_A,

            "loglik_B":
                loglik_B,

            "loglik_C":
                loglik_C,

            "loglik_D":
                loglik_D,

            "p_A":
                p_A,

            "p_B":
                p_B,

            "p_C":
                p_C,

            "p_D":
                p_D,

            "p_correct":
                p_correct,

            "accuracy":
                accuracy,

            "num_fewshot":
                num_fewshot,

            "temperature":
                temperature
        })


# ============================================================
# 9. DataFrame化
# ============================================================

result = pd.DataFrame(
    all_rows
)


# ============================================================
# 10. 検証
# ============================================================

print()
print("==============================")
print("RESULT SUMMARY")
print("==============================")

print(
    "Rows:",
    len(result)
)

print(
    "Subjects:",
    result["subject"].nunique()
)

print(
    "Models:",
    result["model"].nunique()
)

print(
    "Accuracy:",
    result["accuracy"].mean()
)


# ------------------------------------
# pred == gold とaccuracyの一致確認
# ------------------------------------

reconstructed_accuracy = (
    result["pred"]
    ==
    result["gold"]
).astype(int)

agreement = (
    reconstructed_accuracy
    ==
    result["accuracy"]
).mean()

print(
    "Accuracy agreement:",
    agreement
)

mismatch = result[
    reconstructed_accuracy
    != result["accuracy"]
]

print("Mismatch:", len(mismatch))
# ------------------------------------
# subject別問題数
# ------------------------------------

print()
print("Questions per subject:")

print(
    result
    .groupby("subject")
    .size()
    .sort_index()
)


# ============================================================
# 11. CSV保存
# ============================================================

result.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print()
print(
    "Saved:",
    OUTPUT_CSV
)


# ============================================================
# 12. 先頭確認
# ============================================================

print()
print(
    result.head(20).to_string(
        index=False
    )
)

MMLU parquet files found: 57
Number of subjects: 57
['abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge', 'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics', 'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic', 'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science', 'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics', 'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics', 'high_school_physics', 'high_school_psychology', 'high_school_statistics', 'high_school_us_history', 'high_school_world_history', 'human_aging', 'human_sexuality', 'international_law', 'jurisprudence', 'logical_fallacies', 'machine_learning', 'management', 'marketing', 'medical_genetics', 'miscellaneous', 'moral_disputes', '

In [18]:
result["reconstructed_accuracy"] = (
    result["pred"] == result["gold"]
).astype(int)

mismatch = result[
    result["accuracy"]
    != result["reconstructed_accuracy"]
].copy()

print("Mismatch:", len(mismatch))

print(
    mismatch[
        [
            "subject",
            "question",
            "gold",
            "pred",
            "accuracy",
            "p_correct"
        ]
    ].to_string(index=False)
)

Mismatch: 38
                   subject                                                                                                                                                                                                                                                                                                                                                          question gold pred  accuracy  p_correct
                 astronomy                                                                                                                                                                                                                                                                                                         The terrestrial planet cores contain mostly metal because    C    D       1.0   0.001325
                 astronomy                                                                                                                                         

In [20]:
from datasets import load_dataset
import pandas as pd


# ============================================================
# 1. Open LLM Leaderboard v1 のcontentsを取得
# ============================================================

ds = load_dataset(
    "open-llm-leaderboard-old/contents",
    split="train"
)

df = ds.to_pandas()


# ============================================================
# 2. 基本情報
# ============================================================

print("Leaderboard rows:", len(df))
print("Unique fullname:", df["fullname"].nunique())
print("Unique model SHA:", df["Model sha"].nunique())

print()
print(df.columns.tolist())


# ============================================================
# 3. 全評価エントリを保存
# ============================================================

df.to_csv(
    "OpenLLM_v1_all_entries.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: OpenLLM_v1_all_entries.csv")

README.md:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

C:\Users\masta\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\masta\.cache\huggingface\hub\datasets--open-llm-leaderboard-old--contents. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001-96886cb34a7bc8(…): reconstructing file:   0%|          |  0.00B / 1.37MB            

data/train-00000-of-00001-96886cb34a7bc8(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7260 [00:00<?, ? examples/s]

Leaderboard rows: 7260
Unique fullname: 6896
Unique model SHA: 6903

['eval_name', 'Precision', 'Type', 'T', 'Weight type', 'Architecture', 'Model', 'fullname', 'Model sha', 'Average ⬆️', 'Hub License', 'Hub ❤️', '#Params (B)', 'Available on the hub', 'Merged', 'MoE', 'Flagged', 'date', 'Chat Template', 'ARC', 'HellaSwag', 'MMLU', 'TruthfulQA', 'Winogrande', 'GSM8K', 'Maintainers Choice']
Saved: OpenLLM_v1_all_entries.csv


In [21]:
unique_models = (
    df[
        [
            "fullname",
            "Architecture",
            "#Params (B)",
            "Type",
            "Hub License",
            "Available on the hub"
        ]
    ]
    .drop_duplicates(subset=["fullname"])
    .sort_values("fullname")
    .reset_index(drop=True)
)

print("Unique models:", len(unique_models))

unique_models.to_csv(
    "OpenLLM_v1_unique_models.csv",
    index=False,
    encoding="utf-8-sig"
)

print(unique_models.head(30))

Unique models: 6896
                                  fullname        Architecture  #Params (B)  \
0                     0-hero/Matter-0.1-7B  MistralForCausalLM            7   
1         0-hero/Matter-0.1-7B-DPO-preview  MistralForCausalLM            7   
2               0-hero/Matter-0.1-7B-boost  MistralForCausalLM            7   
3           0-hero/Matter-0.1-7B-boost-DPO  MistralForCausalLM            7   
4   0-hero/Matter-0.1-7B-boost-DPO-preview  MistralForCausalLM            7   
5                0-hero/Matter-0.1-Slim-7B  MistralForCausalLM            7   
6              0-hero/Matter-0.1-Slim-7B-A  MistralForCausalLM            7   
7              0-hero/Matter-0.1-Slim-7B-B  MistralForCausalLM            7   
8              0-hero/Matter-0.1-Slim-7B-C  MistralForCausalLM            7   
9          0-hero/Matter-0.1-Slim-7B-C-DPO  MistralForCausalLM            7   
10       0-hero/Matter-0.1-Slim-7B-preview  MistralForCausalLM            7   
11                   0-hero/Matt

In [22]:
eval_models = (
    df[
        [
            "eval_name",
            "fullname",
            "Model sha",
            "Precision",
            "Architecture",
            "#Params (B)",
            "Type",
            "Weight type",
            "Available on the hub",
            "Merged",
            "MoE",
            "Flagged",
            "date",
            "MMLU",
            "Average ⬆️"
        ]
    ]
    .drop_duplicates()
    .sort_values(["fullname", "Model sha", "Precision"])
    .reset_index(drop=True)
)

print("Evaluation configurations:", len(eval_models))

eval_models.to_csv(
    "OpenLLM_v1_model_evaluations.csv",
    index=False,
    encoding="utf-8-sig"
)

Evaluation configurations: 7260


In [23]:
mmlu_models = df[
    df["MMLU"].notna()
].copy()

print("Rows with MMLU:", len(mmlu_models))
print(
    "Unique fullnames with MMLU:",
    mmlu_models["fullname"].nunique()
)

mmlu_models.to_csv(
    "OpenLLM_v1_models_with_MMLU.csv",
    index=False,
    encoding="utf-8-sig"
)

Rows with MMLU: 7260
Unique fullnames with MMLU: 6896


In [24]:
def make_details_repo(fullname):
    return (
        "open-llm-leaderboard-old/details_"
        + fullname.replace("/", "__")
    )


mmlu_models["details_repo"] = (
    mmlu_models["fullname"]
    .apply(make_details_repo)
)

print(
    mmlu_models[
        [
            "fullname",
            "Model sha",
            "Precision",
            "MMLU",
            "details_repo"
        ]
    ].head(20)
)

                                  fullname  \
0                     0-hero/Matter-0.1-7B   
1         0-hero/Matter-0.1-7B-DPO-preview   
2               0-hero/Matter-0.1-7B-boost   
3           0-hero/Matter-0.1-7B-boost-DPO   
4   0-hero/Matter-0.1-7B-boost-DPO-preview   
5                0-hero/Matter-0.1-Slim-7B   
6              0-hero/Matter-0.1-Slim-7B-A   
7              0-hero/Matter-0.1-Slim-7B-B   
8              0-hero/Matter-0.1-Slim-7B-C   
9          0-hero/Matter-0.1-Slim-7B-C-DPO   
10       0-hero/Matter-0.1-Slim-7B-preview   
11                   0-hero/Matter-0.2-32B   
12                    0-hero/Matter-0.2-7B   
13                0-hero/Matter-0.2-7B-DPO   
14                        01-ai/Yi-1.5-34B   
15                    01-ai/Yi-1.5-34B-32K   
16                   01-ai/Yi-1.5-34B-Chat   
17               01-ai/Yi-1.5-34B-Chat-16K   
18                         01-ai/Yi-1.5-6B   
19                         01-ai/Yi-1.5-6B   

                                 

In [25]:
from huggingface_hub import HfApi

api = HfApi()

# archive組織に存在するdataset repositoryを取得
datasets = api.list_datasets(
    author="open-llm-leaderboard-old"
)

dataset_ids = {
    x.id
    for x in datasets
}

print("Datasets in archive:", len(dataset_ids))


# details repoの存在確認
mmlu_models["details_exists"] = (
    mmlu_models["details_repo"]
    .isin(dataset_ids)
)

print(
    mmlu_models["details_exists"].value_counts()
)

Datasets in archive: 7059
details_exists
True     7183
False      77
Name: count, dtype: int64


In [26]:
downloadable = mmlu_models[
    mmlu_models["details_exists"]
].copy()

print(
    "Evaluation rows with details:",
    len(downloadable)
)

print(
    "Unique models with details:",
    downloadable["fullname"].nunique()
)

downloadable.to_csv(
    "OpenLLM_v1_MMLU_details_available.csv",
    index=False,
    encoding="utf-8-sig"
)

Evaluation rows with details: 7183
Unique models with details: 6819


In [29]:
from pathlib import Path, PurePosixPath
from collections import defaultdict
from io import BytesIO
import hashlib
import ast
import re
import math

import numpy as np
import pandas as pd
import requests

from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_url

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ============================================================
# 0. 基本設定
# ============================================================

# Jupyter Notebook の現在の作業ディレクトリ
NOTEBOOK_DIR = Path.cwd()

# ユーザーが作成済みのフォルダ
BASE_DIR = NOTEBOOK_DIR / "260829 MMLU data"

MODEL_LIST_DIR = BASE_DIR / "model_lists"
MODEL_CSV_DIR = BASE_DIR / "model_csv"
DIAGNOSTIC_DIR = BASE_DIR / "diagnostics"

BASE_DIR.mkdir(exist_ok=True)
MODEL_LIST_DIR.mkdir(exist_ok=True)
MODEL_CSV_DIR.mkdir(exist_ok=True)
DIAGNOSTIC_DIR.mkdir(exist_ok=True)

print("Notebook working directory:")
print(NOTEBOOK_DIR)

print("\nOutput directory:")
print(BASE_DIR)


# ============================================================
# 1. どのモデル集合を実行するか
# ============================================================

# 1 = 全6,819モデル
# 2 = 公式providerのみ
# 3 = pretrained / continuously pretrained のみ
# 4 = 代表的な主要base + instruct/chat
SELECTION_MODE = 1


# True にすると既存CSVも再計算
OVERWRITE = True

# 動作確認したい場合だけ整数にする
# 例: MAX_MODELS = 3
# 全件なら None
MAX_MODELS = None


# ============================================================
# 2. 元のモデル一覧ファイルを探す
# ============================================================

candidate_files = [
    NOTEBOOK_DIR / "OpenLLM_v1_MMLU_details_available.csv",
    NOTEBOOK_DIR / "OpenLLM_v1_MMLU_details_available.xls",
    BASE_DIR / "OpenLLM_v1_MMLU_details_available.csv",
    BASE_DIR / "OpenLLM_v1_MMLU_details_available.xls",
]

MODEL_INDEX_PATH = None

for p in candidate_files:
    if p.exists():
        MODEL_INDEX_PATH = p
        break

if MODEL_INDEX_PATH is None:
    raise FileNotFoundError(
        "OpenLLM_v1_MMLU_details_available.csv / .xls が見つかりません。"
    )

print("Model index:", MODEL_INDEX_PATH)


# ============================================================
# 3. CSV / Excel どちらでも読む
#
# .xlsという拡張子でも実体がCSVの場合に対応
# ============================================================

def read_table_auto(path):

    # まずCSVとして試す
    for enc in ["utf-8-sig", "utf-8", "cp932"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass

    # 本物のExcelならこちら
    try:
        return pd.read_excel(path)
    except Exception as e:
        raise RuntimeError(
            f"Could not read model list: {path}"
        ) from e


model_index = read_table_auto(MODEL_INDEX_PATH)

print("Raw evaluation entries:", len(model_index))
print("Unique fullnames:", model_index["fullname"].nunique())


# ============================================================
# 4. fullnameごとに1モデルにする
#
# 同じfullnameに複数precision/revisionがある場合、
# 最新のLeaderboard entryを代表として採用する
# ============================================================

model_index["__date"] = pd.to_datetime(
    model_index["date"],
    errors="coerce",
    utc=True
)

old_date = pd.Timestamp("1900-01-01", tz="UTC")

model_index["__date_sort"] = (
    model_index["__date"]
    .fillna(old_date)
)

unique_models = (
    model_index
    .sort_values(["fullname", "__date_sort"])
    .drop_duplicates(
        subset=["fullname"],
        keep="last"
    )
    .reset_index(drop=True)
)

print("Unique models after deduplication:", len(unique_models))


# ============================================================
# 5. 公式provider namespace
#
# 「公式provider」はLeaderboardに専用flagがないため、
# namespace whitelistで定義する。
# 必要に応じて追加・削除してください。
# ============================================================

OFFICIAL_PROVIDERS = {
    "meta-llama",
    "google",
    "Qwen",
    "mistralai",
    "microsoft",
    "01-ai",
    "deepseek-ai",
    "tiiuae",
    "EleutherAI",
    "bigscience",
    "allenai",
    "databricks",
    "mosaicml",
    "stabilityai",
    "CohereForAI",
    "THUDM",
    "internlm",
    "BAAI",
    "openbmb",
    "bigcode",
    "Salesforce",
    "state-spaces",
    "RWKV",
}


def get_provider(fullname):
    return str(fullname).split("/")[0]


unique_models["provider"] = (
    unique_models["fullname"]
    .apply(get_provider)
)


# ============================================================
# 6. 代表的な主要モデル
#
# v1時代の主要base / instruct / chatを明示的に指定。
# リストに存在するものだけが自動的に採用される。
# ============================================================

MAJOR_MODELS = [

    # -------------------------
    # Meta Llama
    # -------------------------
    "meta-llama/Llama-2-7b-hf",
    "meta-llama/Llama-2-7b-chat-hf",
    "meta-llama/Llama-2-13b-hf",
    "meta-llama/Llama-2-13b-chat-hf",
    "meta-llama/Llama-2-70b-hf",
    "meta-llama/Llama-2-70b-chat-hf",

    "meta-llama/Meta-Llama-3-8B",
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "meta-llama/Meta-Llama-3-70B",
    "meta-llama/Meta-Llama-3-70B-Instruct",

    # -------------------------
    # Google Gemma
    # -------------------------
    "google/gemma-2b",
    "google/gemma-2b-it",
    "google/gemma-7b",
    "google/gemma-7b-it",
    "google/gemma-1.1-2b-it",
    "google/gemma-1.1-7b-it",
    "google/recurrentgemma-2b",
    "google/recurrentgemma-2b-it",

    # -------------------------
    # Mistral
    # -------------------------
    "mistralai/Mistral-7B-v0.1",
    "mistralai/Mistral-7B-Instruct-v0.1",
    "mistralai/Mistral-7B-Instruct-v0.2",
    "mistralai/Mistral-7B-v0.3",
    "mistralai/Mistral-7B-Instruct-v0.3",

    "mistralai/Mixtral-8x7B-v0.1",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "mistralai/Mixtral-8x22B-v0.1",
    "mistralai/Mixtral-8x22B-Instruct-v0.1",

    # -------------------------
    # Qwen 1.5
    # -------------------------
    "Qwen/Qwen1.5-4B",
    "Qwen/Qwen1.5-4B-Chat",
    "Qwen/Qwen1.5-7B",
    "Qwen/Qwen1.5-7B-Chat",
    "Qwen/Qwen1.5-14B",
    "Qwen/Qwen1.5-14B-Chat",
    "Qwen/Qwen1.5-32B",
    "Qwen/Qwen1.5-32B-Chat",
    "Qwen/Qwen1.5-72B",
    "Qwen/Qwen1.5-72B-Chat",

    # -------------------------
    # Qwen 2
    # -------------------------
    "Qwen/Qwen2-0.5B",
    "Qwen/Qwen2-0.5B-Instruct",
    "Qwen/Qwen2-1.5B",
    "Qwen/Qwen2-1.5B-Instruct",
    "Qwen/Qwen2-7B",
    "Qwen/Qwen2-7B-Instruct",
    "Qwen/Qwen2-72B",
    "Qwen/Qwen2-72B-Instruct",

    # -------------------------
    # Microsoft Phi
    # -------------------------
    "microsoft/phi-2",
    "microsoft/Phi-3-mini-4k-instruct",
    "microsoft/Phi-3-mini-128k-instruct",
    "microsoft/Phi-3-small-8k-instruct",
    "microsoft/Phi-3-medium-4k-instruct",
    "microsoft/Phi-3-medium-128k-instruct",

    # -------------------------
    # Yi
    # -------------------------
    "01-ai/Yi-1.5-6B",
    "01-ai/Yi-1.5-6B-Chat",
    "01-ai/Yi-1.5-9B",
    "01-ai/Yi-1.5-9B-Chat",
    "01-ai/Yi-1.5-34B",
    "01-ai/Yi-1.5-34B-Chat",

    # -------------------------
    # DeepSeek
    # -------------------------
    "deepseek-ai/deepseek-llm-7b-base",
    "deepseek-ai/deepseek-llm-7b-chat",
    "deepseek-ai/deepseek-llm-67b-base",
    "deepseek-ai/deepseek-llm-67b-chat",

    # -------------------------
    # Falcon
    # -------------------------
    "tiiuae/falcon-7b",
    "tiiuae/falcon-7b-instruct",
    "tiiuae/falcon-40b",
    "tiiuae/falcon-40b-instruct",
    "tiiuae/falcon-180B",
    "tiiuae/falcon-180B-chat",
]


# ============================================================
# 7. 4種類のモデル集合を作る
# ============================================================

model_lists = {}


# ① 全モデル
model_lists[1] = unique_models.copy()


# ② 公式provider
model_lists[2] = (
    unique_models[
        unique_models["provider"].isin(
            OFFICIAL_PROVIDERS
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ③ pretrained / continuously pretrained
type_text = (
    unique_models["Type"]
    .fillna("")
    .astype(str)
)

model_lists[3] = (
    unique_models[
        type_text.str.contains(
            "pretrained",
            case=False,
            regex=False
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ④ 主要代表モデル
model_lists[4] = (
    unique_models[
        unique_models["fullname"].isin(
            MAJOR_MODELS
        )
    ]
    .copy()
    .reset_index(drop=True)
)


MODE_NAMES = {
    1: "all_unique_models",
    2: "official_providers",
    3: "pretrained_and_continuously_pretrained",
    4: "major_representative_models",
}


# ============================================================
# 8. Windows-safe filename
# ============================================================

def safe_model_filename(fullname):

    s = str(fullname).replace("/", "__")

    s = re.sub(
        r'[<>:"/\\|?*]',
        "_",
        s
    )

    s = s.strip().rstrip(".")

    # 衝突防止
    h = hashlib.sha1(
        str(fullname).encode("utf-8")
    ).hexdigest()[:8]

    # Windowsの長すぎるpath対策
    s = s[:150]

    return f"{s}__{h}.csv"


# ============================================================
# 9. 4種類すべてのモデルリストを保存
# ============================================================

for mode, mdf in model_lists.items():

    mdf = mdf.copy()

    mdf["output_csv"] = (
        mdf["fullname"]
        .apply(safe_model_filename)
    )

    output_list = (
        MODEL_LIST_DIR
        / f"{mode}_{MODE_NAMES[mode]}.csv"
    )

    mdf.to_csv(
        output_list,
        index=False,
        encoding="utf-8-sig"
    )

    model_lists[mode] = mdf

    print(
        f"Mode {mode}: "
        f"{len(mdf)} models -> {output_list.name}"
    )


# ④で指定したが実データに存在しないものも確認
found_major = set(
    model_lists[4]["fullname"]
)

not_found_major = [
    x for x in MAJOR_MODELS
    if x not in found_major
]

print("\nMajor models not found in v1 list:")
for x in not_found_major:
    print("  ", x)


# ============================================================
# 10. MMLU 57分野
# ============================================================

MMLU_SUBJECTS = [
    "abstract_algebra",
    "anatomy",
    "astronomy",
    "business_ethics",
    "clinical_knowledge",
    "college_biology",
    "college_chemistry",
    "college_computer_science",
    "college_mathematics",
    "college_medicine",
    "college_physics",
    "computer_security",
    "conceptual_physics",
    "econometrics",
    "electrical_engineering",
    "elementary_mathematics",
    "formal_logic",
    "global_facts",
    "high_school_biology",
    "high_school_chemistry",
    "high_school_computer_science",
    "high_school_european_history",
    "high_school_geography",
    "high_school_government_and_politics",
    "high_school_macroeconomics",
    "high_school_mathematics",
    "high_school_microeconomics",
    "high_school_physics",
    "high_school_psychology",
    "high_school_statistics",
    "high_school_us_history",
    "high_school_world_history",
    "human_aging",
    "human_sexuality",
    "international_law",
    "jurisprudence",
    "logical_fallacies",
    "machine_learning",
    "management",
    "marketing",
    "medical_genetics",
    "miscellaneous",
    "moral_disputes",
    "moral_scenarios",
    "nutrition",
    "philosophy",
    "prehistory",
    "professional_accounting",
    "professional_law",
    "professional_medicine",
    "professional_psychology",
    "public_relations",
    "security_studies",
    "sociology",
    "us_foreign_policy",
    "virology",
    "world_religions",
]


def normalize_text(s):
    return re.sub(
        r"\s+",
        " ",
        str(s)
    ).strip()


# ============================================================
# 11. MMLU 14,042問のマスターテーブルを一度だけ作る
#
# これを左側に置いて各モデルをmergeすることで、
# 欠測があっても14,042行を必ず維持する
# ============================================================

master_rows = []

for subject in MMLU_SUBJECTS:

    print("Loading MMLU master:", subject)

    ds = load_dataset(
        "cais/mmlu",
        subject,
        split="test"
    )

    for item_index, item in enumerate(ds):

        question = item["question"].strip()
        choices = list(item["choices"])
        gold_index = int(item["answer"])

        formatted = (
            f"{question}\n"
            f"A. {choices[0]}\n"
            f"B. {choices[1]}\n"
            f"C. {choices[2]}\n"
            f"D. {choices[3]}\n"
            f"Answer:"
        )

        master_rows.append({
            "subject": subject,
            "item_index": item_index,
            "item_id":
                f"{subject}__{item_index:04d}",
            "question": question,
            "gold_index": gold_index,
            "gold": "ABCD"[gold_index],
            "_match_key": normalize_text(formatted),
        })


MMLU_MASTER = pd.DataFrame(master_rows)


# 完全同一問題が複数回存在する場合にも対応
MMLU_MASTER["_occurrence"] = (
    MMLU_MASTER
    .groupby(
        ["subject", "_match_key"]
    )
    .cumcount()
)


print("\nMMLU master rows:", len(MMLU_MASTER))
print(
    "MMLU subjects:",
    MMLU_MASTER["subject"].nunique()
)

assert len(MMLU_MASTER) == 14042
assert MMLU_MASTER["subject"].nunique() == 57


# 保存しておくと後で非常に便利
MMLU_MASTER[
    [
        "item_id",
        "subject",
        "item_index",
        "question",
        "gold",
        "gold_index",
    ]
].to_csv(
    BASE_DIR / "MMLU_master_14042_items.csv",
    index=False,
    encoding="utf-8-sig"
)

Notebook working directory:
C:\Users\masta\LLM IRT

Output directory:
C:\Users\masta\LLM IRT\260829 MMLU data
Model index: C:\Users\masta\LLM IRT\OpenLLM_v1_MMLU_details_available.csv
Raw evaluation entries: 7183
Unique fullnames: 6819
Unique models after deduplication: 6819
Mode 1: 6819 models -> 1_all_unique_models.csv
Mode 2: 216 models -> 2_official_providers.csv
Mode 3: 425 models -> 3_pretrained_and_continuously_pretrained.csv
Mode 4: 57 models -> 4_major_representative_models.csv

Major models not found in v1 list:
   google/gemma-1.1-2b-it
   mistralai/Mistral-7B-Instruct-v0.3
   Qwen/Qwen2-0.5B-Instruct
   Qwen/Qwen2-1.5B-Instruct
   Qwen/Qwen2-7B-Instruct
   Qwen/Qwen2-72B-Instruct
   microsoft/Phi-3-small-8k-instruct
   deepseek-ai/deepseek-llm-7b-base
   tiiuae/falcon-40b-instruct
   tiiuae/falcon-180B-chat
Loading MMLU master: abstract_algebra
Loading MMLU master: anatomy
Loading MMLU master: astronomy
Loading MMLU master: business_ethics
Loading MMLU master: clinical_know

In [30]:
# ============================================================
# 12. Hugging Face API / HTTP session
# ============================================================

api = HfApi()

retry = Retry(
    total=5,
    connect=5,
    read=5,
    status=5,
    backoff_factor=1.5,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods=["GET"]
)

session = requests.Session()

adapter = HTTPAdapter(
    max_retries=retry
)

session.mount(
    "https://",
    adapter
)


# ============================================================
# 13. MMLUファイル名をparse
# ============================================================

def extract_subject_fewshot(filename):

    m = re.search(
        r"hendrycksTest-(.+?)\|(\d+)_",
        filename
    )

    if m is None:
        return None, None

    subject = m.group(1)
    fewshot = int(m.group(2))

    return subject, fewshot


# ============================================================
# 14. repo内のMMLU runを列挙
# ============================================================

def get_mmlu_runs(repo_id):

    files = api.list_repo_files(
        repo_id=repo_id,
        repo_type="dataset"
    )

    runs = defaultdict(
        lambda: defaultdict(list)
    )

    for f in files:

        if (
            "hendrycksTest-" not in f
            or not f.endswith(".parquet")
        ):
            continue

        subject, fewshot = (
            extract_subject_fewshot(f)
        )

        if subject not in MMLU_SUBJECTS:
            continue

        parent = (
            PurePosixPath(f)
            .parent
            .as_posix()
        )

        if parent == ".":
            parent = "__root__"

        runs[parent][subject].append(
            (f, fewshot)
        )

    return runs


# ============================================================
# 15. run timestamp
# ============================================================

def parse_run_datetime(run_id):

    if run_id == "__root__":
        return pd.NaT

    basename = (
        PurePosixPath(run_id).name
    )

    return pd.to_datetime(
        basename,
        errors="coerce",
        utc=True
    )


# ============================================================
# 16. モデルリストのdateに最も近いrunを選ぶ
#
# 同じfullnameに複数runが存在するため重要
# ============================================================

def choose_best_run(
    runs,
    target_date
):

    if len(runs) == 0:
        return None, {}, 0

    target_date = pd.to_datetime(
        target_date,
        errors="coerce",
        utc=True
    )

    candidates = []

    for run_id, subject_map in runs.items():

        dt = parse_run_datetime(run_id)

        subject_count = len(subject_map)

        if (
            pd.notna(target_date)
            and pd.notna(dt)
        ):
            delta = abs(
                (
                    dt - target_date
                ).total_seconds()
            )
        else:
            delta = math.inf

        candidates.append({
            "run_id": run_id,
            "datetime": dt,
            "subject_count": subject_count,
            "delta": delta,
        })


    # evaluation dateとの距離を最優先
    dated = [
        x for x in candidates
        if np.isfinite(x["delta"])
    ]

    if len(dated) > 0:

        chosen = sorted(
            dated,
            key=lambda x: (
                x["delta"],
                -x["subject_count"]
            )
        )[0]

    else:

        # 日付が使えない場合：
        # subject数最大 → 最新run
        chosen = sorted(
            candidates,
            key=lambda x: (
                -x["subject_count"],
                (
                    x["datetime"]
                    if pd.notna(x["datetime"])
                    else old_date
                )
            ),
            reverse=False
        )[0]


    run_id = chosen["run_id"]

    # 同一subjectに複数fileがあれば最後のもの
    selected_files = {}

    for subject, candidates_for_subject in (
        runs[run_id].items()
    ):

        candidates_for_subject = sorted(
            candidates_for_subject,
            key=lambda x: x[0]
        )

        selected_files[subject] = (
            candidates_for_subject[-1]
        )

    return (
        run_id,
        selected_files,
        chosen["subject_count"]
    )


# ============================================================
# 17. Windowsに保存せずParquetを読む
# ============================================================

def load_remote_parquet(
    repo_id,
    filename
):

    url = hf_hub_url(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset"
    )

    r = session.get(
        url,
        timeout=(20, 180)
    )

    r.raise_for_status()

    return pd.read_parquet(
        BytesIO(r.content)
    )


# ============================================================
# 18. objectをparse
# ============================================================

def parse_obj(x):

    if isinstance(x, str):

        try:
            return ast.literal_eval(x)

        except Exception:
            return x

    return x


# ============================================================
# 19. predictionsから4つのlog-likelihoodを抽出
#
# 以下の両方に対応
#
# [-1.2, -2.3, ...]
#
# [(score, greedy), ...]
# ============================================================

def extract_four_scores(x):

    x = parse_obj(x)

    if isinstance(x, np.ndarray):
        x = x.tolist()

    if isinstance(x, dict):

        if "result" in x:
            x = x["result"]
        else:
            return None

    if not isinstance(
        x,
        (list, tuple)
    ):
        return None

    if len(x) != 4:
        return None

    values = []

    for item in x:

        item = parse_obj(item)

        if isinstance(
            item,
            (int, float, np.integer, np.floating)
        ):

            values.append(float(item))
            continue


        if isinstance(item, dict):

            if "result" not in item:
                return None

            item = item["result"]


        if isinstance(
            item,
            np.ndarray
        ):
            item = item.tolist()


        if isinstance(
            item,
            (list, tuple)
        ):

            if len(item) == 0:
                return None

            first = item[0]

            if isinstance(
                first,
                (
                    int,
                    float,
                    np.integer,
                    np.floating
                )
            ):
                values.append(
                    float(first)
                )
                continue

        return None


    arr = np.asarray(
        values,
        dtype=float
    )

    if np.isnan(arr).any():
        return None

    return arr


# ============================================================
# 20. stable softmax
# ============================================================

def softmax4(x):

    x = np.asarray(
        x,
        dtype=float
    )

    if np.isnan(x).any():
        return None

    if not np.isfinite(x).any():
        return None

    m = np.max(
        x[np.isfinite(x)]
    )

    z = np.exp(x - m)

    total = z.sum()

    if (
        not np.isfinite(total)
        or total <= 0
    ):
        return None

    return z / total


# ============================================================
# 21. metricsからaccuracyを取得
# ============================================================

def extract_accuracy(metrics):

    metrics = parse_obj(metrics)

    if isinstance(metrics, dict):

        if "acc" in metrics:
            try:
                return float(
                    metrics["acc"]
                )
            except Exception:
                pass

    return np.nan


# ============================================================
# 22. fewshot
# ============================================================

def extract_fewshot(
    row,
    fallback
):

    try:
        value = row[
            "num_effective_few_shots"
        ]

        if pd.notna(value):
            return int(value)

    except Exception:
        pass

    return fallback


# ============================================================
# 23. 1 subject のParquetを処理
# ============================================================
def process_subject_parquet(
    repo_id,
    filename,
    subject,
    filename_fewshot
):

    df = load_remote_parquet(
        repo_id,
        filename
    )

    # ========================================================
    # 重要：
    # dfと同じindexを持つDataFrameを最初から作る
    # ========================================================

    temp = pd.DataFrame(
        index=df.index
    )

    temp["subject"] = subject

    temp["_match_key"] = (
        df["example"]
        .apply(normalize_text)
    )

    # 同一問題が複数あった場合用
    temp["_occurrence"] = (
        temp
        .groupby(
            ["subject", "_match_key"]
        )
        .cumcount()
    )

    temp["_detail_present"] = True


    records = []

    for _, row in df.iterrows():

        scores = extract_four_scores(
            row.get("predictions")
        )

        if scores is None:

            rec = {
                "loglik_A": np.nan,
                "loglik_B": np.nan,
                "loglik_C": np.nan,
                "loglik_D": np.nan,
                "p_A": np.nan,
                "p_B": np.nan,
                "p_C": np.nan,
                "p_D": np.nan,
                "pred": np.nan,
                "accuracy":
                    extract_accuracy(
                        row.get("metrics")
                    ),
                "num_fewshot":
                    extract_fewshot(
                        row,
                        filename_fewshot
                    ),
                "data_status":
                    "invalid_predictions",
            }

            records.append(rec)
            continue


        probs = softmax4(scores)

        if probs is None:

            records.append({
                "loglik_A": scores[0],
                "loglik_B": scores[1],
                "loglik_C": scores[2],
                "loglik_D": scores[3],
                "p_A": np.nan,
                "p_B": np.nan,
                "p_C": np.nan,
                "p_D": np.nan,
                "pred": np.nan,
                "accuracy":
                    extract_accuracy(
                        row.get("metrics")
                    ),
                "num_fewshot":
                    extract_fewshot(
                        row,
                        filename_fewshot
                    ),
                "data_status":
                    "invalid_softmax",
            })

            continue


        pred_idx = int(
            np.argmax(scores)
        )

        records.append({

            "loglik_A": scores[0],
            "loglik_B": scores[1],
            "loglik_C": scores[2],
            "loglik_D": scores[3],

            "p_A": probs[0],
            "p_B": probs[1],
            "p_C": probs[2],
            "p_D": probs[3],

            "pred":
                "ABCD"[pred_idx],

            "accuracy":
                extract_accuracy(
                    row.get("metrics")
                ),

            "num_fewshot":
                extract_fewshot(
                    row,
                    filename_fewshot
                ),

            "data_status":
                "observed",
        })


    values = pd.DataFrame(
        records,
        index=df.index
    )

    temp = pd.concat(
        [
            temp,
            values
        ],
        axis=1
    )

    return temp.reset_index(drop=True)

# ============================================================
# 24. 1モデルを14,042行にする
# ============================================================

def process_one_model(model_row):

    model = str(
        model_row["fullname"]
    )

    repo_id = str(
        model_row["details_repo"]
    )

    target_date = model_row.get(
        "date",
        None
    )


    # ------------------------
    # MMLU run検索
    # ------------------------

    runs = get_mmlu_runs(
        repo_id
    )

    run_id, run_files, n_subjects = (
        choose_best_run(
            runs,
            target_date
        )
    )


    technical_errors = []

    observed_parts = []


    # ------------------------
    # details自体がない場合
    # ------------------------

    if run_id is not None:

        for subject, (
            filename,
            filename_fewshot
        ) in sorted(
            run_files.items()
        ):

            try:

                part = (
                    process_subject_parquet(
                        repo_id=repo_id,
                        filename=filename,
                        subject=subject,
                        filename_fewshot=
                            filename_fewshot
                    )
                )

                observed_parts.append(
                    part
                )

            except Exception as e:

                technical_errors.append(
                    {
                        "model": model,
                        "subject": subject,
                        "filename": filename,
                        "error":
                            repr(e)
                    }
                )


    if len(observed_parts) > 0:

        observed = pd.concat(
            observed_parts,
            ignore_index=True
        )

    else:

        observed = pd.DataFrame(
            columns=[
                "subject",
                "_match_key",
                "_occurrence",
                "_detail_present",
                "loglik_A",
                "loglik_B",
                "loglik_C",
                "loglik_D",
                "p_A",
                "p_B",
                "p_C",
                "p_D",
                "pred",
                "accuracy",
                "num_fewshot",
                "data_status",
            ]
        )


    # ========================================================
    # details側にあるがMMLU masterと対応しない行を確認
    # ========================================================

    canonical_keys = (
        MMLU_MASTER[
            [
                "subject",
                "_match_key",
                "_occurrence"
            ]
        ]
        .drop_duplicates()
    )

    obs_check = observed.merge(
        canonical_keys,
        on=[
            "subject",
            "_match_key",
            "_occurrence"
        ],
        how="left",
        indicator=True
    )

    unmatched_detail_rows = int(
        (
            obs_check["_merge"]
            == "left_only"
        ).sum()
    )


    # ========================================================
    # MASTERを左にしてmerge
    #
    # → 欠測しても14,042行を維持
    # ========================================================

    merged = MMLU_MASTER.merge(
        observed,
        on=[
            "subject",
            "_match_key",
            "_occurrence"
        ],
        how="left"
    )


    # ------------------------
    # missing status
    # ------------------------

    merged["data_status"] = (
        merged["data_status"]
        .fillna("missing")
    )


    # ------------------------
    # p_correct
    # ------------------------

    merged["p_correct"] = np.nan

    prob_cols = [
        "p_A",
        "p_B",
        "p_C",
        "p_D"
    ]

    for gold_index in range(4):

        mask = (
            merged["gold_index"]
            == gold_index
        )

        merged.loc[
            mask,
            "p_correct"
        ] = merged.loc[
            mask,
            prob_cols[gold_index]
        ]


    # ========================================================
    # 出力テーブル
    # ========================================================

    out = pd.DataFrame({

        "model":
            model,

        "subject":
            merged["subject"],

        # 後のモデル間join用。
        # 強く推奨する追加列
        "item_id":
            merged["item_id"],

        "question":
            merged["question"],

        "gold":
            merged["gold"],

        "pred":
            merged["pred"],

        "loglik_A":
            merged["loglik_A"],

        "loglik_B":
            merged["loglik_B"],

        "loglik_C":
            merged["loglik_C"],

        "loglik_D":
            merged["loglik_D"],

        "p_A":
            merged["p_A"],

        "p_B":
            merged["p_B"],

        "p_C":
            merged["p_C"],

        "p_D":
            merged["p_D"],

        "p_correct":
            merged["p_correct"],

        "accuracy":
            merged["accuracy"],

        "num_fewshot":
            merged["num_fewshot"],

        # MMLU log-likelihood評価では非該当
        "temperature":
            np.nan,

        # 欠測確認用
        "data_status":
            merged["data_status"],
    })


    # ========================================================
    # validation
    # ========================================================

    comparable = (
        out["pred"].notna()
        &
        out["accuracy"].notna()
    )

    if comparable.sum() > 0:

        reconstructed = (
            out.loc[
                comparable,
                "pred"
            ]
            ==
            out.loc[
                comparable,
                "gold"
            ]
        ).astype(float)

        agreement = float(
            (
                reconstructed
                ==
                out.loc[
                    comparable,
                    "accuracy"
                ]
            ).mean()
        )

        mismatch = int(
            (
                reconstructed
                !=
                out.loc[
                    comparable,
                    "accuracy"
                ]
            ).sum()
        )

    else:

        agreement = np.nan
        mismatch = 0


    observed_rows = int(
        (
            out["data_status"]
            == "observed"
        ).sum()
    )

    missing_rows = int(
        (
            out["data_status"]
            == "missing"
        ).sum()
    )

    invalid_rows = int(
        len(out)
        -
        observed_rows
        -
        missing_rows
    )


    summary = {

        "model":
            model,

        "repo_id":
            repo_id,

        "selected_run":
            run_id,

        "subjects_in_run":
            n_subjects,

        "total_rows":
            len(out),

        "observed_rows":
            observed_rows,

        "missing_rows":
            missing_rows,

        "invalid_rows":
            invalid_rows,

        "unmatched_detail_rows":
            unmatched_detail_rows,

        "accuracy_mean":
            out["accuracy"].mean(),

        "accuracy_agreement":
            agreement,

        "mismatch":
            mismatch,

        "technical_errors":
            len(technical_errors),
    }


    return (
        out,
        summary,
        technical_errors
    )


# ============================================================
# 25. diagnostics保存
# ============================================================

def append_processing_log(
    summary,
    log_path
):

    temp = pd.DataFrame(
        [summary]
    )

    temp.to_csv(
        log_path,
        mode="a",
        header=not log_path.exists(),
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 26. 選択したモデル集合を処理
# ============================================================

selected_models = (
    model_lists[
        SELECTION_MODE
    ]
    .copy()
    .reset_index(drop=True)
)

if MAX_MODELS is not None:

    selected_models = (
        selected_models
        .head(MAX_MODELS)
        .copy()
    )


MODE_NAME = (
    MODE_NAMES[
        SELECTION_MODE
    ]
)

PROCESSING_LOG = (
    DIAGNOSTIC_DIR
    / f"processing_log_{SELECTION_MODE}_{MODE_NAME}.csv"
)

ERROR_LOG = (
    DIAGNOSTIC_DIR
    / f"errors_{SELECTION_MODE}_{MODE_NAME}.csv"
)


print()
print("==============================")
print("PROCESSING START")
print("==============================")
print("Mode:", SELECTION_MODE)
print("Mode name:", MODE_NAME)
print("Models:", len(selected_models))
print()


for i, model_row in (
    selected_models.iterrows()
):

    model = str(
        model_row["fullname"]
    )

    filename = (
        safe_model_filename(
            model
        )
    )

    final_path = (
        MODEL_CSV_DIR
        / filename
    )

    partial_path = (
        MODEL_CSV_DIR
        / filename.replace(
            ".csv",
            ".partial.csv"
        )
    )


    print(
        f"[{i + 1}/{len(selected_models)}] "
        f"{model}"
    )


    # ------------------------
    # 完成済みはスキップ
    # ------------------------

    if (
        final_path.exists()
        and not OVERWRITE
    ):

        print(
            "  -> already completed, skip"
        )

        continue


    try:

        out, summary, errors = (
            process_one_model(
                model_row
            )
        )


        # 14,042行であることを強制
        assert len(out) == 14042


        # ====================================================
        # technical errorやvalidation mismatchがある場合は
        # .partial.csv として保存
        #
        # → 次回runで完成品扱いされない
        # ====================================================

        has_technical_error = (
            summary["technical_errors"] > 0
        )

        has_validation_error = (
            summary["mismatch"] > 0
        )

        has_matching_error = (
            summary["unmatched_detail_rows"] > 0
        )

        has_no_observations = (
            summary["observed_rows"] == 0
        )

        if (
            has_technical_error
            or has_validation_error
            or has_matching_error
            or has_no_observations
        ):
            save_path = partial_path
            print("  -> PARTIAL")
        else:
            save_path = final_path

            # 古いpartialがあれば消す
            if partial_path.exists():
                partial_path.unlink()

            print(
                "  -> COMPLETE"
            )


        out.to_csv(
            save_path,
            index=False,
            encoding="utf-8-sig",
            float_format="%.12g"
        )


        append_processing_log(
            summary,
            PROCESSING_LOG
        )


        # subject単位の通信エラー等
        if len(errors) > 0:

            err_df = pd.DataFrame(
                errors
            )

            err_df.to_csv(
                ERROR_LOG,
                mode="a",
                header=not ERROR_LOG.exists(),
                index=False,
                encoding="utf-8-sig"
            )


        print(
            "     run:",
            summary["selected_run"]
        )

        print(
            "     subjects:",
            summary["subjects_in_run"]
        )

        print(
            "     observed:",
            summary["observed_rows"]
        )

        print(
            "     missing:",
            summary["missing_rows"]
        )

        print(
            "     invalid:",
            summary["invalid_rows"]
        )

        print(
            "     accuracy:",
            summary["accuracy_mean"]
        )

        print(
            "     agreement:",
            summary["accuracy_agreement"]
        )

        print(
            "     mismatch:",
            summary["mismatch"]
        )


    except Exception as e:

        print(
            "  -> MODEL ERROR:",
            repr(e)
        )

        error_row = pd.DataFrame([
            {
                "model": model,
                "subject": "",
                "filename": "",
                "error":
                    repr(e)
            }
        ])

        error_row.to_csv(
            ERROR_LOG,
            mode="a",
            header=not ERROR_LOG.exists(),
            index=False,
            encoding="utf-8-sig"
        )


PROCESSING START
Mode: 1
Mode name: all_unique_models
Models: 6819

[1/6819] 0-hero/Matter-0.1-7B
  -> COMPLETE
     run: 2024-03-22T00-43-45.003432
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.6162227602905569
     agreement: 1.0
     mismatch: 0
[2/6819] 0-hero/Matter-0.1-7B-DPO-preview
  -> COMPLETE
     run: 2024-03-23T05-48-47.699955
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.6186440677966102
     agreement: 1.0
     mismatch: 0
[3/6819] 0-hero/Matter-0.1-7B-boost
  -> COMPLETE
     run: 2024-03-22T00-38-52.022465
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.612021079618288
     agreement: 1.0
     mismatch: 0
[4/6819] 0-hero/Matter-0.1-7B-boost-DPO
  -> COMPLETE
     run: 2024-03-22T17-52-36.450234
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.6117362199116935
     agreement: 1.0
     mismatch: 0
[5/6819] 0-h

C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[


  -> PARTIAL
     run: None
     subjects: 0
     observed: 0
     missing: 14042
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[30/6819] 01-ai/Yi-9B
  -> COMPLETE
     run: 2024-03-07T00-53-16.402231
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.6848027346531833
     agreement: 1.0
     mismatch: 0
[31/6819] 01-ai/Yi-9B-200K
  -> COMPLETE
     run: 2024-03-21T16-22-28.590236
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.6892892750320467
     agreement: 1.0
     mismatch: 0
[32/6819] 0ai/0ai-7B-v3
  -> COMPLETE
     run: 2024-05-01T06-13-40.016291
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.5710724967953283
     agreement: 1.0
     mismatch: 0
[33/6819] 0ai/0ai-7B-v4
  -> COMPLETE
     run: 2024-05-02T02-39-04.562402
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.5213644779945876
     agreement

C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[


  -> PARTIAL
     run: None
     subjects: 0
     observed: 0
     missing: 14042
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[186/6819] Aeala/Enterredaas-33b


...

  -> PARTIAL
     run: None
     subjects: 0
     observed: 0
     missing: 14042
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[6790/6819] zarakiquemparte/zararp-1.1-l2-7b
  -> COMPLETE
     run: 2023-10-01T15-07-29.187841
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[6791/6819] zarakiquemparte/zararp-l2-7b
  -> COMPLETE
     run: 2023-09-12T16-17-38.915453
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[6792/6819] zarakiquemparte/zaraxe-l2-7b
  -> COMPLETE
     run: 2023-08-23T21:46:04.335707
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[6793/6819] zarakiquemparte/zaraxls-l2-7b
  -> COMPLETE
     run: 2023-08-28T20:28:21.792080
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: nan
  

C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[


  -> PARTIAL
     run: None
     subjects: 0
     observed: 0
     missing: 14042
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[6811/6819] ziqingyang/chinese-alpaca-2-7b
  -> COMPLETE
     run: 2023-08-09T11:39:32.814142
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[6812/6819] ziqingyang/chinese-llama-2-13b


C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[


  -> PARTIAL
     run: None
     subjects: 0
     observed: 0
     missing: 14042
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[6813/6819] ziqingyang/chinese-llama-2-7b
  -> COMPLETE
     run: 2023-08-09T11:36:32.525773
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: nan
     agreement: nan
     mismatch: 0
[6814/6819] zmzmxz/NeuralPipe-7B-slerp
  -> COMPLETE
     run: 2024-04-16T01-54-06.237773
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.6089588377723971
     agreement: 1.0
     mismatch: 0
[6815/6819] zorobin/mistral-class-shishya-7b-ep3
  -> COMPLETE
     run: 2024-01-28T05-38-18.308889
     subjects: 57
     observed: 14042
     missing: 0
     invalid: 0
     accuracy: 0.38975929354792765
     agreement: 1.0
     mismatch: 0
[6816/6819] zorobin/mistral-class-shishya-all-hal-7b-ep3
  -> COMPLETE
     run: 2024-01-28T05-47-57.937695
     subjects: 57
     observed: 14042
    

In [31]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from contextlib import ExitStack
import csv
import hashlib
import os
import re

import numpy as np
import pandas as pd


# ============================================================
# 0. 設定
# ============================================================

BASE_DIR = Path.cwd() / "260829 MMLU data"

MODEL_CSV_DIR = (
    BASE_DIR / "model_csv"
)

MODEL_LIST_PATH = (
    BASE_DIR
    / "model_lists"
    / "1_all_unique_models.csv"
)

MMLU_MASTER_PATH = (
    BASE_DIR
    / "MMLU_master_14042_items.csv"
)

OUTPUT_DIR = (
    BASE_DIR
    / "p_correct_matrices_by_subject"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 並列数
#
# SSDなら 8 程度から開始推奨
# ============================================================

MAX_WORKERS = 8


print("Model CSV directory:")
print(MODEL_CSV_DIR)

print("\nOutput directory:")
print(OUTPUT_DIR)


# ============================================================
# 1. モデル一覧
#
# この順番を57分野すべての共通行順とする
# ============================================================

models = pd.read_csv(
    MODEL_LIST_PATH,
    encoding="utf-8-sig"
)


if "fullname" not in models.columns:
    raise ValueError(
        "model list に fullname 列がありません。"
    )


if models["fullname"].duplicated().any():

    dup = models.loc[
        models["fullname"].duplicated(
            keep=False
        ),
        "fullname"
    ]

    raise ValueError(
        "fullname に重複があります:\n"
        + "\n".join(
            dup.astype(str).head(20)
        )
    )


MODEL_NAMES = (
    models["fullname"]
    .astype(str)
    .tolist()
)


print(
    "Number of models:",
    len(MODEL_NAMES)
)


# ============================================================
# 2. 前回と同じsafe filename
# ============================================================

def safe_model_filename(fullname):

    s = str(fullname).replace(
        "/",
        "__"
    )

    s = re.sub(
        r'[<>:"/\\|?*]',
        "_",
        s
    )

    s = s.strip().rstrip(".")

    h = hashlib.sha1(
        str(fullname).encode("utf-8")
    ).hexdigest()[:8]

    s = s[:150]

    return (
        f"{s}__{h}.csv"
    )


# モデルリストにoutput_csvがある場合は
# それを最優先で使う
if "output_csv" in models.columns:

    MODEL_FILENAMES = (
        models["output_csv"]
        .astype(str)
        .tolist()
    )

else:

    MODEL_FILENAMES = [
        safe_model_filename(x)
        for x in MODEL_NAMES
    ]


# ============================================================
# 3. MMLU 14,042問マスター
# ============================================================

master = pd.read_csv(
    MMLU_MASTER_PATH,
    encoding="utf-8-sig"
)


required_master_cols = {
    "item_id",
    "subject",
    "item_index",
    "question",
    "gold",
}

missing_cols = (
    required_master_cols
    - set(master.columns)
)

if missing_cols:
    raise ValueError(
        f"MMLU masterに必要列がありません: "
        f"{missing_cols}"
    )


master = master.reset_index(
    drop=True
)


if master["item_id"].duplicated().any():
    raise ValueError(
        "MMLU masterのitem_idに重複があります。"
    )


print(
    "MMLU items:",
    len(master)
)

print(
    "MMLU subjects:",
    master["subject"].nunique()
)


assert len(master) == 14042
assert master["subject"].nunique() == 57


# ============================================================
# 4. item_id -> MMLU全体での位置
# ============================================================

ITEM_TO_POSITION = pd.Series(
    np.arange(
        len(master),
        dtype=np.int32
    ),
    index=master["item_id"].astype(str)
)


# ============================================================
# 5. 分野ごとの問題順を固定
# ============================================================

SUBJECTS = sorted(
    master["subject"]
    .unique()
    .tolist()
)


SUBJECT_ITEM_IDS = {}
SUBJECT_POSITIONS = {}
SUBJECT_METADATA = {}


for subject in SUBJECTS:

    sm = (
        master[
            master["subject"]
            == subject
        ]
        .sort_values(
            "item_index"
        )
        .copy()
    )

    SUBJECT_ITEM_IDS[
        subject
    ] = (
        sm["item_id"]
        .astype(str)
        .tolist()
    )

    SUBJECT_POSITIONS[
        subject
    ] = (
        sm.index
        .to_numpy(
            dtype=np.int32
        )
    )

    SUBJECT_METADATA[
        subject
    ] = sm


print(
    "\nSubjects prepared:",
    len(SUBJECTS)
)


# ============================================================
# 6. 各分野の問題対応表を保存
#
# 行列の各列がどの問題か確認するため
# ============================================================

for subject in SUBJECTS:

    meta = (
        SUBJECT_METADATA[
            subject
        ][
            [
                "item_id",
                "item_index",
                "question",
                "gold",
            ]
        ]
    )

    meta.to_csv(
        OUTPUT_DIR
        / f"MMLU_items_{subject}.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 7. モデル順マスター
#
# 57分野すべてこの順番
# ============================================================

pd.DataFrame({
    "model_order":
        np.arange(
            1,
            len(MODEL_NAMES) + 1
        ),

    "model":
        MODEL_NAMES,

}).to_csv(
    OUTPUT_DIR
    / "MMLU_model_order.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 8. モデルファイルを探す
#
# COMPLETEを優先
# なければpartial
# それもなければ全問欠測
# ============================================================

def find_model_file(filename):

    final_path = (
        MODEL_CSV_DIR
        / filename
    )

    if final_path.exists():

        return (
            final_path,
            "complete_file"
        )


    partial_path = (
        MODEL_CSV_DIR
        / filename.replace(
            ".csv",
            ".partial.csv"
        )
    )

    if partial_path.exists():

        return (
            partial_path,
            "partial_file"
        )


    return (
        None,
        "missing_file"
    )


# ============================================================
# 9. 1モデルを読むworker
#
# 14,042要素の共通vectorに
# p_correctを配置する
#
# CSV内の行順には依存しない。
# item_idによって正しい問題位置に配置する。
# ============================================================

def read_one_model(args):

    model_no, model_name, filename = args


    # --------------------------------------------------------
    # 14,042問のp_correct
    #
    # 欠測は np.nan
    # --------------------------------------------------------

    pcorrect_vector = np.full(
        len(master),
        np.nan,
        dtype=np.float64
    )


    path, file_status = (
        find_model_file(
            filename
        )
    )


    # ========================================================
    # ファイル自体がない場合
    #
    # モデル行は削除せず、全問NaNで返す
    # ========================================================

    if path is None:

        return {
            "model_no":
                model_no,

            "model":
                model_name,

            "file_status":
                file_status,

            "pcorrect_vector":
                pcorrect_vector,

            "n_rows_read":
                0,

            "n_valid_p_correct":
                0,

            "n_missing_p_correct":
                len(master),

            "n_invalid_p_correct":
                0,

            "n_duplicate_item_id":
                0,

            "n_unknown_item_id":
                0,

            "error":
                "",
        }


    try:

        # ====================================================
        # 必要な列だけ読む
        # ====================================================

        df = pd.read_csv(
            path,
            usecols=[
                "item_id",
                "p_correct"
            ],
            dtype={
                "item_id": "string"
            },
            encoding="utf-8-sig"
        )


        n_rows_read = len(df)


        # ====================================================
        # item_id重複確認
        # ====================================================

        n_duplicate = int(
            df["item_id"]
            .duplicated()
            .sum()
        )


        if n_duplicate > 0:

            # 同じ問題を二重投入しない
            df = (
                df
                .drop_duplicates(
                    subset="item_id",
                    keep="first"
                )
                .copy()
            )


        # ====================================================
        # item_id -> master上の位置
        # ====================================================

        positions = (
            df["item_id"]
            .astype(str)
            .map(
                ITEM_TO_POSITION
            )
        )


        n_unknown_item_id = int(
            positions.isna().sum()
        )


        known = (
            positions.notna()
        )


        df_known = (
            df.loc[
                known
            ]
            .copy()
        )


        positions_known = (
            positions.loc[
                known
            ]
            .astype(np.int32)
            .to_numpy()
        )


        # ====================================================
        # p_correctを数値に変換
        # ====================================================

        p_correct = (
            pd.to_numeric(
                df_known[
                    "p_correct"
                ],
                errors="coerce"
            )
        )


        # ====================================================
        # 0 <= p_correct <= 1 だけを正常値とする
        # ====================================================

        valid = (
            p_correct.notna()
            &
            (p_correct >= 0.0)
            &
            (p_correct <= 1.0)
        )


        n_valid = int(
            valid.sum()
        )


        # NaNは単純なmissing
        n_missing = int(
            p_correct.isna().sum()
        )


        # 数値だが0～1外はinvalid
        n_invalid = int(
            (
                p_correct.notna()
                &
                ~(
                    (p_correct >= 0.0)
                    &
                    (p_correct <= 1.0)
                )
            ).sum()
        )


        # ====================================================
        # 正常値だけmaster位置へ配置
        # ====================================================

        valid_positions = (
            positions_known[
                valid.to_numpy()
            ]
        )


        valid_values = (
            p_correct.loc[
                valid
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        pcorrect_vector[
            valid_positions
        ] = valid_values


        # masterに存在するが
        # このモデルにそもそも行がない問題も含めた
        # 最終的な欠測数
        total_missing = int(
            np.isnan(
                pcorrect_vector
            ).sum()
        )


        return {

            "model_no":
                model_no,

            "model":
                model_name,

            "file_status":
                file_status,

            "pcorrect_vector":
                pcorrect_vector,

            "n_rows_read":
                n_rows_read,

            "n_valid_p_correct":
                n_valid,

            "n_missing_p_correct":
                total_missing,

            "n_invalid_p_correct":
                n_invalid,

            "n_duplicate_item_id":
                n_duplicate,

            "n_unknown_item_id":
                n_unknown_item_id,

            "error":
                "",
        }


    except Exception as e:

        # ====================================================
        # 読み込みエラーでもモデルを落とさない
        #
        # 全問NaNの行として維持
        # ====================================================

        return {

            "model_no":
                model_no,

            "model":
                model_name,

            "file_status":
                "read_error",

            "pcorrect_vector":
                pcorrect_vector,

            "n_rows_read":
                0,

            "n_valid_p_correct":
                0,

            "n_missing_p_correct":
                len(master),

            "n_invalid_p_correct":
                0,

            "n_duplicate_item_id":
                0,

            "n_unknown_item_id":
                0,

            "error":
                repr(e),
        }


# ============================================================
# 10. worker入力
# ============================================================

jobs = [

    (
        i,
        MODEL_NAMES[i],
        MODEL_FILENAMES[i]
    )

    for i in range(
        len(MODEL_NAMES)
    )
]


# ============================================================
# 11. 57個の行列CSVを同時に作る
#
# executor.map はjobsと同じ順番で結果を返す
#
# → 並列処理しても57分野すべて同じモデル順
# ============================================================

diagnostics = []


# 実行途中は tmp.csv
# 最後まで成功した場合のみ正式名へrename
tmp_paths = {}


for subject in SUBJECTS:

    tmp_paths[
        subject
    ] = (
        OUTPUT_DIR
        / f"MMLU_p_correct_{subject}.tmp.csv"
    )


with ExitStack() as stack:

    writers = {}


    # ========================================================
    # 57個のCSVを開き、headerを作る
    # ========================================================

    for subject in SUBJECTS:

        fh = stack.enter_context(
            open(
                tmp_paths[
                    subject
                ],
                "w",
                newline="",
                encoding="utf-8-sig"
            )
        )


        writer = csv.writer(
            fh,
            lineterminator="\n"
        )


        # 1列目model
        # 2列目以降はitem_id
        writer.writerow(
            ["model"]
            +
            SUBJECT_ITEM_IDS[
                subject
            ]
        )


        writers[
            subject
        ] = writer


    # ========================================================
    # モデルCSVを並列読み込み
    # ========================================================

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        results = executor.map(
            read_one_model,
            jobs
        )


        for done_no, result in enumerate(
            results,
            start=1
        ):

            model_name = (
                result["model"]
            )

            full_vector = (
                result[
                    "pcorrect_vector"
                ]
            )


            # =================================================
            # 57分野それぞれに書き込む
            # =================================================

            for subject in SUBJECTS:

                positions = (
                    SUBJECT_POSITIONS[
                        subject
                    ]
                )


                subject_values = (
                    full_vector[
                        positions
                    ]
                )


                # ---------------------------------------------
                # np.nanをCSVでは空欄にする
                # ---------------------------------------------

                csv_values = [

                    ""
                    if np.isnan(x)
                    else format(
                        float(x),
                        ".12g"
                    )

                    for x in subject_values
                ]


                writers[
                    subject
                ].writerow(
                    [model_name]
                    +
                    csv_values
                )


            # =================================================
            # 診断情報
            # =================================================

            diagnostics.append({

                k: v

                for k, v
                in result.items()

                if k !=
                "pcorrect_vector"
            })


            # =================================================
            # 進捗
            # =================================================

            if (
                done_no % 100 == 0
                or
                done_no == len(jobs)
            ):

                print(
                    f"{done_no}"
                    f" / "
                    f"{len(jobs)}"
                    f" models processed"
                )


# ============================================================
# 12. tmp -> 正式CSV
# ============================================================

for subject in SUBJECTS:

    final_path = (
        OUTPUT_DIR
        / f"MMLU_p_correct_{subject}.csv"
    )


    os.replace(
        tmp_paths[
            subject
        ],
        final_path
    )


print(
    "\n57 p_correct matrices created."
)


# ============================================================
# 13. diagnostics
# ============================================================

diagnostics_df = pd.DataFrame(
    diagnostics
)


diagnostics_df.to_csv(
    OUTPUT_DIR
    / "MMLU_p_correct_matrix_diagnostics.csv",
    index=False,
    encoding="utf-8-sig"
)


print()
print("==============================")
print("FILE STATUS")
print("==============================")

print(
    diagnostics_df[
        "file_status"
    ]
    .value_counts(
        dropna=False
    )
)


print()
print(
    "Models with no valid p_correct:"
)

print(
    (
        diagnostics_df[
            "n_valid_p_correct"
        ]
        == 0
    ).sum()
)


print()
print(
    "Models with some missing p_correct:"
)

print(
    (
        diagnostics_df[
            "n_missing_p_correct"
        ]
        > 0
    ).sum()
)


# ============================================================
# 14. 各分野CSVを検証
# ============================================================

subject_summary = []


expected_models = pd.Series(
    MODEL_NAMES,
    name="model"
)


for subject in SUBJECTS:

    path = (
        OUTPUT_DIR
        / f"MMLU_p_correct_{subject}.csv"
    )


    # --------------------------------------------------------
    # header
    # --------------------------------------------------------

    header = pd.read_csv(
        path,
        nrows=0,
        encoding="utf-8-sig"
    ).columns.tolist()


    expected_header = (
        ["model"]
        +
        SUBJECT_ITEM_IDS[
            subject
        ]
    )


    columns_ok = (
        header
        ==
        expected_header
    )


    # --------------------------------------------------------
    # model順
    # --------------------------------------------------------

    model_col = pd.read_csv(
        path,
        usecols=[
            "model"
        ],
        encoding="utf-8-sig"
    )[
        "model"
    ].astype(str)


    model_order_ok = (

        len(model_col)
        ==
        len(expected_models)

        and

        model_col
        .reset_index(
            drop=True
        )
        .equals(
            expected_models
            .reset_index(
                drop=True
            )
        )
    )


    subject_summary.append({

        "subject":
            subject,

        "n_models":
            len(model_col),

        "n_items":
            len(
                SUBJECT_ITEM_IDS[
                    subject
                ]
            ),

        "columns_ok":
            columns_ok,

        "model_order_ok":
            model_order_ok,
    })


subject_summary_df = pd.DataFrame(
    subject_summary
)


subject_summary_df.to_csv(
    OUTPUT_DIR
    / "MMLU_p_correct_subject_validation.csv",
    index=False,
    encoding="utf-8-sig"
)


print()
print(
    subject_summary_df.to_string(
        index=False
    )
)


# ============================================================
# 15. 最終validation
# ============================================================

assert (
    subject_summary_df[
        "n_models"
    ]
    ==
    len(MODEL_NAMES)
).all()


assert (
    subject_summary_df[
        "columns_ok"
    ]
).all()


assert (
    subject_summary_df[
        "model_order_ok"
    ]
).all()


print()
print("==============================")
print("ALL VALIDATIONS PASSED")
print("==============================")

print(
    "Subjects:",
    len(SUBJECTS)
)

print(
    "Models per subject:",
    len(MODEL_NAMES)
)

Model CSV directory:
C:\Users\masta\LLM IRT\260829 MMLU data\model_csv

Output directory:
C:\Users\masta\LLM IRT\260829 MMLU data\p_correct_matrices_by_subject
Number of models: 6819
MMLU items: 14042
MMLU subjects: 57

Subjects prepared: 57
100 / 6819 models processed
200 / 6819 models processed
300 / 6819 models processed
400 / 6819 models processed
500 / 6819 models processed
600 / 6819 models processed
700 / 6819 models processed
800 / 6819 models processed
900 / 6819 models processed
1000 / 6819 models processed
1100 / 6819 models processed
1200 / 6819 models processed
1300 / 6819 models processed
1400 / 6819 models processed
1500 / 6819 models processed
1600 / 6819 models processed
1700 / 6819 models processed
1800 / 6819 models processed
1900 / 6819 models processed
2000 / 6819 models processed
2100 / 6819 models processed
2200 / 6819 models processed
2300 / 6819 models processed
2400 / 6819 models processed
2500 / 6819 models processed
2600 / 6819 models processed
2700 / 6819 m

In [32]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd() / "260829 MMLU data"

MODEL_LIST_PATH = (
    BASE_DIR
    / "model_lists"
    / "1_all_unique_models.csv"
)

MODEL_CSV_DIR = BASE_DIR / "model_csv"

DIAG_PATH = (
    BASE_DIR
    / "p_correct_matrices_by_subject"
    / "MMLU_p_correct_matrix_diagnostics.csv"
)

diag = pd.read_csv(
    DIAG_PATH,
    encoding="utf-8-sig"
)

models = pd.read_csv(
    MODEL_LIST_PATH,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 「完全に正常」なのは
# 14,042問すべてに有効なp_correctがあるモデル
# ------------------------------------------------------------

bad_mask = (
    (diag["n_valid_p_correct"] != 14042)
    |
    (diag["n_missing_p_correct"] > 0)
    |
    (diag["n_invalid_p_correct"] > 0)
    |
    (diag["file_status"] != "complete_file")
)

bad_diag = (
    diag.loc[bad_mask]
    .copy()
    .reset_index(drop=True)
)

bad_names = set(
    bad_diag["model"].astype(str)
)

retry_models = (
    models[
        models["fullname"]
        .astype(str)
        .isin(bad_names)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Models to repair:", len(retry_models))

display(
    bad_diag[
        [
            "model",
            "file_status",
            "n_valid_p_correct",
            "n_missing_p_correct",
            "n_invalid_p_correct",
            "error",
        ]
    ]
)

bad_diag.to_csv(
    BASE_DIR / "retry_targets_before.csv",
    index=False,
    encoding="utf-8-sig"
)

Models to repair: 346


,model,file_status,n_valid_p_correct,n_missing_p_correct,n_invalid_p_correct,error
0,01-ai/Yi-6B-200K,partial_file,0,14042,0,NaN
1,APMIC/caigun-lora-model-33B,partial_file,0,14042,0,NaN
2,AbacusResearch/jaLLAbi,partial_file,0,14042,0,NaN
3,Aeala/Alpaca-elina-65b,partial_file,0,14042,0,NaN
4,Aeala/Enterredaas-33b,partial_file,0,14042,0,NaN
...,...,...,...,...,...,...
341,yeontaek/llama-2-13b-QLoRA,partial_file,0,14042,0,NaN
342,zarakiquemparte/zarablend-l2-7b,partial_file,0,14042,0,NaN
343,zarakiquemparte/zarafusionix-l2-7b,partial_file,0,14042,0,NaN
344,ziqingyang/chinese-alpaca-2-13b,partial_file,0,14042,0,NaN


In [ ]:
import time
import random

from huggingface_hub import HfApi
from huggingface_hub.errors import (
    RepositoryNotFoundError,
    GatedRepoError,
    HfHubHTTPError,
)

from requests.exceptions import (
    Timeout,
    ConnectionError,
)


def list_repo_files_retry(
    repo_id,
    max_attempts=6,
    base_wait=2.0,
):

    api_local = HfApi()

    for attempt in range(max_attempts):

        try:

            return api_local.list_repo_files(
                repo_id=repo_id,
                repo_type="dataset",
            )

        # ====================================================
        # permanent error
        # → retryしない
        # ====================================================

        except RepositoryNotFoundError:
            print(
                f"repo not found: {repo_id}"
            )
            return None

        except GatedRepoError:
            print(
                f"gated repo: {repo_id}"
            )
            return None

        # ====================================================
        # HTTP error
        # ====================================================

        except HfHubHTTPError as e:

            status = None

            try:
                status = e.response.status_code
            except Exception:
                pass

            # 401 / 403 / 404 は待っても直らない
            if status in (401, 403, 404):

                print(
                    f"permanent HTTP {status}: "
                    f"{repo_id}"
                )

                return None

            # 429 / 5xx はretry
            if (
                status == 429
                or
                (
                    status is not None
                    and 500 <= status < 600
                )
            ):

                if attempt == max_attempts - 1:
                    raise

                wait = (
                    base_wait * (2 ** attempt)
                    + random.uniform(0, 1)
                )

                print(
                    f"retry API {repo_id} "
                    f"after {wait:.1f}s "
                    f"(HTTP {status})"
                )

                time.sleep(wait)
                continue

            raise

        except (
            Timeout,
            ConnectionError
        ):

            if attempt == max_attempts - 1:
                raise

            wait = (
                base_wait * (2 ** attempt)
                + random.uniform(0, 1)
            )

            print(
                f"network retry {repo_id} "
                f"after {wait:.1f}s"
            )

            time.sleep(wait)

    return None

In [34]:
def choose_best_run(
    runs,
    target_date
):

    if len(runs) == 0:
        return None, {}, 0

    target_date = pd.to_datetime(
        target_date,
        errors="coerce",
        utc=True,
    )

    candidates = []

    for run_id, subject_map in runs.items():

        dt = parse_run_datetime(
            run_id
        )

        subject_count = len(
            subject_map
        )

        if (
            pd.notna(target_date)
            and
            pd.notna(dt)
        ):
            delta = abs(
                (
                    dt - target_date
                ).total_seconds()
            )
        else:
            delta = np.inf

        candidates.append({
            "run_id": run_id,
            "datetime": dt,
            "subject_count": subject_count,
            "delta": delta,
        })

    # ========================================================
    # 最重要：
    # まず57 subject揃っているrunを優先
    # ========================================================

    complete_runs = [
        x for x in candidates
        if x["subject_count"] == 57
    ]

    if complete_runs:
        pool = complete_runs
    else:
        max_subjects = max(
            x["subject_count"]
            for x in candidates
        )

        pool = [
            x for x in candidates
            if x["subject_count"]
            == max_subjects
        ]

    # ========================================================
    # 同程度の完全性ならLeaderboard dateに近いrun
    # ========================================================

    dated = [
        x for x in pool
        if np.isfinite(x["delta"])
    ]

    if dated:

        chosen = min(
            dated,
            key=lambda x: x["delta"],
        )

    else:

        # 日付比較不能なら最新
        old_date = pd.Timestamp(
            "1900-01-01",
            tz="UTC"
        )

        chosen = max(
            pool,
            key=lambda x: (
                x["datetime"]
                if pd.notna(x["datetime"])
                else old_date
            ),
        )

    run_id = chosen["run_id"]

    selected_files = {}

    for subject, candidates_for_subject in (
        runs[run_id].items()
    ):

        candidates_for_subject = sorted(
            candidates_for_subject,
            key=lambda x: x[0],
        )

        selected_files[subject] = (
            candidates_for_subject[-1]
        )

    return (
        run_id,
        selected_files,
        chosen["subject_count"],
    )

In [35]:
def choose_best_run(
    runs,
    target_date
):

    if len(runs) == 0:
        return None, {}, 0

    target_date = pd.to_datetime(
        target_date,
        errors="coerce",
        utc=True,
    )

    candidates = []

    for run_id, subject_map in runs.items():

        dt = parse_run_datetime(
            run_id
        )

        subject_count = len(
            subject_map
        )

        if (
            pd.notna(target_date)
            and
            pd.notna(dt)
        ):
            delta = abs(
                (
                    dt - target_date
                ).total_seconds()
            )
        else:
            delta = np.inf

        candidates.append({
            "run_id": run_id,
            "datetime": dt,
            "subject_count": subject_count,
            "delta": delta,
        })

    # ========================================================
    # 最重要：
    # まず57 subject揃っているrunを優先
    # ========================================================

    complete_runs = [
        x for x in candidates
        if x["subject_count"] == 57
    ]

    if complete_runs:
        pool = complete_runs
    else:
        max_subjects = max(
            x["subject_count"]
            for x in candidates
        )

        pool = [
            x for x in candidates
            if x["subject_count"]
            == max_subjects
        ]

    # ========================================================
    # 同程度の完全性ならLeaderboard dateに近いrun
    # ========================================================

    dated = [
        x for x in pool
        if np.isfinite(x["delta"])
    ]

    if dated:

        chosen = min(
            dated,
            key=lambda x: x["delta"],
        )

    else:

        # 日付比較不能なら最新
        old_date = pd.Timestamp(
            "1900-01-01",
            tz="UTC"
        )

        chosen = max(
            pool,
            key=lambda x: (
                x["datetime"]
                if pd.notna(x["datetime"])
                else old_date
            ),
        )

    run_id = chosen["run_id"]

    selected_files = {}

    for subject, candidates_for_subject in (
        runs[run_id].items()
    ):

        candidates_for_subject = sorted(
            candidates_for_subject,
            key=lambda x: x[0],
        )

        selected_files[subject] = (
            candidates_for_subject[-1]
        )

    return (
        run_id,
        selected_files,
        chosen["subject_count"],
    )

In [36]:
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
)

import traceback


RETRY_WORKERS = 3


def candidate_detail_repos(
    fullname,
    original_repo=None,
):

    safe = str(fullname).replace(
        "/",
        "__"
    )

    candidates = []

    if (
        original_repo is not None
        and
        str(original_repo) != "nan"
    ):
        candidates.append(
            str(original_repo)
        )

    candidates += [
        # v1 archive
        f"open-llm-leaderboard-old/details_{safe}",

        # migration前の旧namespace候補
        f"open-llm-leaderboard/details_{safe}",

        # 現在のrepo
        # ※ hendrycksTest/MMLU-v1 filesが実際に
        #    存在するときしか採用しない
        f"open-llm-leaderboard/{safe}-details",
    ]

    # 重複除去
    return list(
        dict.fromkeys(candidates)
    )


def evaluate_output_quality(
    out,
    summary,
):

    p = pd.to_numeric(
        out["p_correct"],
        errors="coerce",
    )

    valid = (
        p.notna()
        &
        p.between(0, 1)
    )

    n_valid = int(
        valid.sum()
    )

    complete = (
        len(out) == 14042
        and
        n_valid == 14042
        and
        summary["technical_errors"] == 0
        and
        summary["mismatch"] == 0
        and
        summary["unmatched_detail_rows"] == 0
    )

    return n_valid, complete


def retry_one_model(model_row):

    fullname = str(
        model_row["fullname"]
    )

    # HFへの同時アクセスを少し散らす
    time.sleep(
        random.uniform(0, 2.0)
    )

    best = None
    attempts = []

    repos = candidate_detail_repos(
        fullname,
        model_row.get(
            "details_repo",
            None
        ),
    )

    for repo_id in repos:

        row2 = model_row.copy()

        row2["details_repo"] = (
            repo_id
        )

        try:

            out, summary, errors = (
                process_one_model(
                    row2
                )
            )

            n_valid, complete = (
                evaluate_output_quality(
                    out,
                    summary,
                )
            )

            attempts.append({
                "model": fullname,
                "repo_id": repo_id,
                "run":
                    summary[
                        "selected_run"
                    ],
                "subjects":
                    summary[
                        "subjects_in_run"
                    ],
                "n_valid":
                    n_valid,
                "missing":
                    summary[
                        "missing_rows"
                    ],
                "invalid":
                    summary[
                        "invalid_rows"
                    ],
                "technical_errors":
                    summary[
                        "technical_errors"
                    ],
                "complete":
                    complete,
                "error": "",
            })

            if (
                best is None
                or
                n_valid > best["n_valid"]
            ):
                best = {
                    "out": out,
                    "summary": summary,
                    "errors": errors,
                    "n_valid": n_valid,
                    "complete": complete,
                    "repo_id": repo_id,
                }

            if complete:
                break

        except Exception as e:

            attempts.append({
                "model": fullname,
                "repo_id": repo_id,
                "run": None,
                "subjects": 0,
                "n_valid": 0,
                "missing": 14042,
                "invalid": 0,
                "technical_errors": 1,
                "complete": False,
                "error": repr(e),
            })

    # ========================================================
    # 保存
    # ========================================================

    filename = safe_model_filename(
        fullname
    )

    final_path = (
        MODEL_CSV_DIR
        / filename
    )

    partial_path = (
        MODEL_CSV_DIR
        / filename.replace(
            ".csv",
            ".partial.csv",
        )
    )

    if best is None:

        return {
            "model": fullname,
            "success": False,
            "n_valid": 0,
            "repo_id": None,
            "attempts": attempts,
        }

    out = best["out"]

    if best["complete"]:

        # atomicに置換
        tmp_path = (
            MODEL_CSV_DIR
            / filename.replace(
                ".csv",
                ".retry.tmp.csv",
            )
        )

        out.to_csv(
            tmp_path,
            index=False,
            encoding="utf-8-sig",
            float_format="%.12g",
        )

        tmp_path.replace(
            final_path
        )

        if partial_path.exists():
            partial_path.unlink()

    else:

        # 完全には直らなくても
        # 最良結果をpartialとして保存
        out.to_csv(
            partial_path,
            index=False,
            encoding="utf-8-sig",
            float_format="%.12g",
        )

        # 元のfinalが「実は不完全」だった場合、
        # 次のmatrix作成でfinalを優先しないようbackup
        old_diag = bad_diag[
            bad_diag["model"]
            .astype(str)
            == fullname
        ]

        if (
            final_path.exists()
            and
            not old_diag.empty
            and
            int(
                old_diag.iloc[0][
                    "n_valid_p_correct"
                ]
            ) < 14042
        ):

            backup = (
                MODEL_CSV_DIR
                / filename.replace(
                    ".csv",
                    ".bad_backup.csv",
                )
            )

            if not backup.exists():
                final_path.replace(
                    backup
                )

    return {
        "model": fullname,
        "success":
            best["complete"],
        "n_valid":
            best["n_valid"],
        "repo_id":
            best["repo_id"],
        "attempts":
            attempts,
    }

In [ ]:
retry_results = []

with ThreadPoolExecutor(
    max_workers=RETRY_WORKERS
) as executor:

    futures = {
        executor.submit(
            retry_one_model,
            row
        ): str(row["fullname"])

        for _, row
        in retry_models.iterrows()
    }

    for i, future in enumerate(
        as_completed(futures),
        start=1,
    ):

        model = futures[future]

        try:

            result = future.result()

        except Exception as e:

            result = {
                "model": model,
                "success": False,
                "n_valid": 0,
                "repo_id": None,
                "attempts": [{
                    "model": model,
                    "error": traceback.format_exc(),
                }],
            }

        retry_results.append(
            result
        )

        print(
            f"[{i}/{len(futures)}] "
            f"{model}: "
            f"success={result['success']} "
            f"valid={result['n_valid']}"
        )

C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[
C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[
C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[


[1/346] 01-ai/Yi-6B-200K: success=False valid=0


C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[
C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[


retry API open-llm-leaderboard/Aeala__Alpaca-elina-65b-details after 3.0s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aeala__Alpaca-elina-65b-details after 4.9s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aeala__Alpaca-elina-65b-details after 8.7s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aeala__Alpaca-elina-65b-details after 16.5s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aeala__Alpaca-elina-65b-details after 32.4s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aeala__Alpaca-elina-65b-details after 64.0s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aeala__Alpaca-elina-65b-details after 128.0s (RepositoryNotFoundError)
retry API open-llm-leaderboard/APMIC__caigun-lora-model-33B-details after 2.6s (RepositoryNotFoundError)
retry API open-llm-leaderboard/AbacusResearch__jaLLAbi-details after 2.3s (RepositoryNotFoundError)
retry API open-llm-leaderboard/AbacusResearch__jaLLAbi-details after 4.8s (RepositoryNotFo

C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[
C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[


...

retry API open-llm-leaderboard/Aspik101__tulu-7b-instruct-pl-lora_unload-details after 3.0s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aspik101__tulu-7b-instruct-pl-lora_unload-details after 4.0s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aspik101__tulu-7b-instruct-pl-lora_unload-details after 8.4s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Andron00e__YetAnother_Open-Llama-3B-LoRA-OpenOrca-details after 128.9s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aspik101__tulu-7b-instruct-pl-lora_unload-details after 16.8s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aspik101__30B-Lazarus-instruct-PL-lora_unload-details after 128.9s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aspik101__tulu-7b-instruct-pl-lora_unload-details after 32.6s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aspik101__tulu-7b-instruct-pl-lora_unload-details after 64.1s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Aspik1

C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[
C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[
C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[


retry API open-llm-leaderboard/Azure99__blossom-v3-mistral-7b-details after 2.6s (RepositoryNotFoundError)


C:\Users\masta\AppData\Local\Temp\ipykernel_13864\3618442872.py:806: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[


retry API open-llm-leaderboard/Azure99__blossom-v1-3b-details after 2.1s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Azure99__blossom-v3-mistral-7b-details after 4.7s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Azure99__blossom-v1-3b-details after 4.4s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Azure99__blossom-v1-3b-details after 8.8s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Azure99__blossom-v3-mistral-7b-details after 8.1s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Azure99__blossom-v3-mistral-7b-details after 16.1s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Azure99__blossom-v1-3b-details after 16.2s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Azure99__blossom-v3-mistral-7b-details after 32.6s (RepositoryNotFoundError)
retry API open-llm-leaderboard/Azure99__blossom-v1-3b-details after 32.4s (RepositoryNotFoundError)


In [15]:
from pathlib import Path
from urllib.parse import quote
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
import numpy as np
import json
import time
import random
import re


from packaging.version import Version
from pathlib import Path
import pandas as pd
import re


# ============================================================
# ここまでのMMLU releaseを全部使う
# ============================================================

TARGET_VERSION = Version("1.13.0")

BUCKET = "crfm-helm-public"

RUNS_PREFIX = (
    "mmlu/benchmark_output/runs/"
)

BASE_DIR = Path(
    "260905 HELM MMLU data"
)

BASE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BASE_DIR = Path("260905 HELM MMLU data")

RAW_DIR = BASE_DIR / "raw_json"
SUBJECT_MATRIX_DIR = (
    BASE_DIR / "correct_matrices_by_subject"
)

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUBJECT_MATRIX_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# MMLU 57 subjects
# ============================================================

MMLU_SUBJECTS = [
    "abstract_algebra",
    "anatomy",
    "astronomy",
    "business_ethics",
    "clinical_knowledge",
    "college_biology",
    "college_chemistry",
    "college_computer_science",
    "college_mathematics",
    "college_medicine",
    "college_physics",
    "computer_security",
    "conceptual_physics",
    "econometrics",
    "electrical_engineering",
    "elementary_mathematics",
    "formal_logic",
    "global_facts",
    "high_school_biology",
    "high_school_chemistry",
    "high_school_computer_science",
    "high_school_european_history",
    "high_school_geography",
    "high_school_government_and_politics",
    "high_school_macroeconomics",
    "high_school_mathematics",
    "high_school_microeconomics",
    "high_school_physics",
    "high_school_psychology",
    "high_school_statistics",
    "high_school_us_history",
    "high_school_world_history",
    "human_aging",
    "human_sexuality",
    "international_law",
    "jurisprudence",
    "logical_fallacies",
    "machine_learning",
    "management",
    "marketing",
    "medical_genetics",
    "miscellaneous",
    "moral_disputes",
    "moral_scenarios",
    "nutrition",
    "philosophy",
    "prehistory",
    "professional_accounting",
    "professional_law",
    "professional_medicine",
    "professional_psychology",
    "public_relations",
    "security_studies",
    "sociology",
    "us_foreign_policy",
    "virology",
    "world_religions",
]

assert len(MMLU_SUBJECTS) == 57


# 多すぎる並列は不要
N_WORKERS = 6

session = requests.Session()

In [16]:
# ============================================================
# Public GCS object listing
# ============================================================

def list_gcs_objects(prefix):
    """
    Public GCS bucket内のobject名をすべて取得。
    ファイル本体はまだDLしない。
    """

    endpoint = (
        "https://storage.googleapis.com/"
        f"storage/v1/b/{BUCKET}/o"
    )

    objects = []
    page_token = None

    while True:

        params = {
            "prefix": prefix,
            "maxResults": 1000,
        }

        if page_token is not None:
            params["pageToken"] = page_token

        r = session.get(
            endpoint,
            params=params,
            timeout=60,
        )

        r.raise_for_status()

        data = r.json()

        for x in data.get("items", []):
            objects.append(x["name"])

        page_token = data.get(
            "nextPageToken"
        )

        if page_token is None:
            break

        print(
            "objects found:",
            len(objects)
        )

    return objects

In [17]:
all_objects = list_gcs_objects(
    GCS_PREFIX
)

print(
    "Total GCS objects:",
    len(all_objects)
)

prediction_objects = [
    x for x in all_objects
    if x.endswith(
        "/display_predictions.json"
    )
]

print(
    "display_predictions files:",
    len(prediction_objects)
)

objects found: 1000
Total GCS objects: 1824
display_predictions files: 228


In [19]:
# ============================================================
# MMLU benchmark_output/runs 以下を全部列挙
# ============================================================

all_objects = list_gcs_objects(
    RUNS_PREFIX
)

print(
    "Total objects under MMLU runs:",
    len(all_objects)
)

objects found: 1000
objects found: 2000
objects found: 3000
objects found: 4000
objects found: 5000
objects found: 6000
objects found: 7000
objects found: 8000
objects found: 9000
objects found: 10000
objects found: 11000
objects found: 12000
objects found: 13000
objects found: 14000
objects found: 15000
objects found: 16000
objects found: 17000
objects found: 18000
objects found: 19000
objects found: 20000
objects found: 21000
objects found: 22000
objects found: 23000
objects found: 24000
objects found: 25000
objects found: 26000
objects found: 27000
objects found: 28000
objects found: 29000
objects found: 30000
objects found: 31000
objects found: 32000
objects found: 33000
objects found: 34000
objects found: 35000
objects found: 36000
Total objects under MMLU runs: 36597


In [20]:
# ============================================================
# GCS上に存在するMMLU suite/versionを取得
# ============================================================

suite_names = set()

for object_name in all_objects:

    relative = object_name[
        len(RUNS_PREFIX):
    ]

    if "/" not in relative:
        continue

    suite = relative.split(
        "/",
        1
    )[0]

    if re.fullmatch(
        r"v\d+\.\d+\.\d+",
        suite
    ):
        suite_names.add(
            suite
        )


suite_names = sorted(
    suite_names,
    key=lambda x:
        Version(
            x.removeprefix("v")
        )
)


print(
    "Available suites:"
)

for x in suite_names:
    print(x)

Available suites:
v1.0.0
v1.1.0
v1.2.0
v1.3.0
v1.4.0
v1.5.0
v1.6.0
v1.7.0
v1.8.0
v1.9.0
v1.10.0
v1.11.0
v1.12.0
v1.13.0


In [21]:
selected_suites = [
    suite
    for suite in suite_names
    if Version(
        suite.removeprefix("v")
    ) <= TARGET_VERSION
]


print(
    "\nSuites included:"
)

for x in selected_suites:
    print(x)

print(
    "\nNumber of suites:",
    len(selected_suites)
)


Suites included:
v1.0.0
v1.1.0
v1.2.0
v1.3.0
v1.4.0
v1.5.0
v1.6.0
v1.7.0
v1.8.0
v1.9.0
v1.10.0
v1.11.0
v1.12.0
v1.13.0

Number of suites: 14


In [23]:
prediction_rows = []

for object_name in all_objects:

    if not object_name.endswith(
        "/display_predictions.json"
    ):
        continue

    relative = object_name[
        len(RUNS_PREFIX):
    ]

    parts = relative.split(
        "/",
        1
    )

    if len(parts) != 2:
        continue

    suite = parts[0]

    if suite not in selected_suites:
        continue

    run_name = (
        parts[1]
        .rsplit("/", 1)[0]
    )

    prediction_rows.append({
        "suite": suite,
        "run_name": run_name,
        "prediction_object":
            object_name,
    })


prediction_manifest = pd.DataFrame(
    prediction_rows
)

print(
    "display_predictions files:",
    len(prediction_manifest)
)

display(
    prediction_manifest.head()
)

display_predictions files: 4574


,suite,run_name,prediction_object
0,v1.0.0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
1,v1.0.0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
2,v1.0.0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
3,v1.0.0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
4,v1.0.0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...


In [9]:
def parse_helm_run_name(run_name):
    """
    HELM run directory名をdictに変換。
    """

    parts = run_name.split(",")

    first = parts[0]

    if ":" not in first:
        return None

    scenario, first_arg = first.split(
        ":",
        1,
    )

    params = {
        "scenario": scenario
    }

    tokens = [first_arg] + parts[1:]

    for token in tokens:

        if "=" not in token:
            continue

        key, value = token.split(
            "=",
            1,
        )

        params[key] = value

    return params

In [24]:
manifest_rows = []

for _, r in prediction_manifest.iterrows():

    suite = r["suite"]
    run_name = r["run_name"]

    params = parse_helm_run_name(
        run_name
    )

    if params is None:
        continue

    if params.get("scenario") != "mmlu":
        continue

    subject = params.get(
        "subject"
    )

    model = params.get(
        "model"
    )

    method = params.get(
        "method"
    )

    if subject not in MMLU_SUBJECTS:
        continue

    if model is None:
        continue

    manifest_rows.append({

        "suite":
            suite,

        "suite_version":
            Version(
                suite.removeprefix("v")
            ),

        "subject":
            subject,

        "model":
            model,

        "method":
            method,

        "run_name":
            run_name,

        "prediction_object":
            r["prediction_object"],

        "eval_split":
            params.get(
                "eval_split"
            ),

        "groups":
            params.get(
                "groups"
            ),

        "additional_instructions":
            params.get(
                "additional_instructions"
            ),
    })


manifest = pd.DataFrame(
    manifest_rows
)


print(
    "All MMLU runs through",
    TARGET_VERSION,
    ":",
    len(manifest)
)

print(
    "Unique models:",
    manifest["model"].nunique()
)

print(
    "Unique subjects:",
    manifest["subject"].nunique()
)

All MMLU runs through 1.13.0 : 4574
Unique models: 81
Unique subjects: 57


In [25]:
suite_summary = (
    manifest
    .groupby("suite")
    .agg(
        n_models=(
            "model",
            "nunique"
        ),
        n_subjects=(
            "subject",
            "nunique"
        ),
        n_runs=(
            "prediction_object",
            "size"
        ),
    )
    .reset_index()
)


suite_summary[
    "_version"
] = (
    suite_summary["suite"]
    .str.removeprefix("v")
    .map(Version)
)

suite_summary = (
    suite_summary
    .sort_values(
        "_version"
    )
    .drop(
        columns="_version"
    )
)

display(
    suite_summary
)

,suite,n_models,n_subjects,n_runs
0,v1.0.0,23,57,1269
1,v1.1.0,6,57,342
6,v1.2.0,6,57,342
7,v1.3.0,3,57,171
8,v1.4.0,10,57,570
9,v1.5.0,1,57,57
10,v1.6.0,5,57,285
11,v1.7.0,6,57,342
12,v1.8.0,4,57,227
13,v1.9.0,4,57,228


In [26]:
suite_summary = (
    manifest
    .groupby("suite")
    .agg(
        n_models=(
            "model",
            "nunique"
        ),
        n_subjects=(
            "subject",
            "nunique"
        ),
        n_runs=(
            "prediction_object",
            "size"
        ),
    )
    .reset_index()
)


suite_summary[
    "_version"
] = (
    suite_summary["suite"]
    .str.removeprefix("v")
    .map(Version)
)

suite_summary = (
    suite_summary
    .sort_values(
        "_version"
    )
    .drop(
        columns="_version"
    )
)

display(
    suite_summary
)

,suite,n_models,n_subjects,n_runs
0,v1.0.0,23,57,1269
1,v1.1.0,6,57,342
6,v1.2.0,6,57,342
7,v1.3.0,3,57,171
8,v1.4.0,10,57,570
9,v1.5.0,1,57,57
10,v1.6.0,5,57,285
11,v1.7.0,6,57,342
12,v1.8.0,4,57,227
13,v1.9.0,4,57,228


In [11]:
standard = manifest.copy()

# standard multiple-choice evaluation
standard = standard[
    standard["method"]
    == "multiple_choice_joint"
].copy()


# additional instruction付きrunは除外
standard = standard[
    standard[
        "additional_instructions"
    ].isna()
].copy()


# eval_split が明示される場合は test のみ
standard = standard[
    standard["eval_split"].isna()
    |
    (
        standard["eval_split"]
        == "test"
    )
].copy()


print(
    "Standard runs:",
    len(standard)
)

print(
    "Models:",
    standard["model"].nunique()
)

print(
    "Subjects:",
    standard["subject"].nunique()
)

Standard runs: 228
Models: 4
Subjects: 57


In [29]:
# Version を並べ替え可能な形にする
standard["_version"] = (
    standard["suite"]
    .str.removeprefix("v")
    .map(Version)
)


# ============================================================
# 同じ model × subject が複数versionにあれば
# 最も新しいversionを使用
# ============================================================

standard = (
    standard
    .sort_values(
        [
            "model",
            "subject",
            "_version",
        ],
        ascending=[
            True,
            True,
            False,
        ]
    )
)


selected = (
    standard
    .drop_duplicates(
        [
            "model",
            "subject",
        ],
        keep="first",
    )
    .drop(
        columns="_version"
    )
    .reset_index(
        drop=True
    )
)


print(
    "Final unique models:",
    selected["model"]
    .nunique()
)

print(
    "Final unique subjects:",
    selected["subject"]
    .nunique()
)

print(
    "Final model-subject pairs:",
    len(selected)
)

KeyError: 'suite'

In [30]:
# ============================================================
# prediction_manifest → manifest → standard → selected
# を全部作り直す
# ============================================================

from packaging.version import Version
import pandas as pd


# ------------------------------------------------------------
# 1. display_predictions.json を version付きで抽出
# ------------------------------------------------------------

prediction_rows = []

for object_name in all_objects:

    if not object_name.endswith(
        "/display_predictions.json"
    ):
        continue

    relative = object_name[
        len(RUNS_PREFIX):
    ]

    parts = relative.split(
        "/",
        1
    )

    if len(parts) != 2:
        continue

    suite = parts[0]

    if suite not in selected_suites:
        continue

    run_name = (
        parts[1]
        .rsplit("/", 1)[0]
    )

    prediction_rows.append({
        "suite": suite,
        "run_name": run_name,
        "prediction_object":
            object_name,
    })


prediction_manifest = pd.DataFrame(
    prediction_rows
)

print(
    "prediction_manifest columns:",
    prediction_manifest.columns.tolist()
)

print(
    "display_predictions files:",
    len(prediction_manifest)
)


# ------------------------------------------------------------
# 2. model / subject 等をparseしてmanifest作成
# ------------------------------------------------------------

manifest_rows = []

for _, r in prediction_manifest.iterrows():

    suite = r["suite"]
    run_name = r["run_name"]

    params = parse_helm_run_name(
        run_name
    )

    if params is None:
        continue

    if params.get("scenario") != "mmlu":
        continue

    subject = params.get(
        "subject"
    )

    model = params.get(
        "model"
    )

    method = params.get(
        "method"
    )

    if subject not in MMLU_SUBJECTS:
        continue

    if model is None:
        continue

    manifest_rows.append({

        "suite":
            suite,

        "subject":
            subject,

        "model":
            model,

        "method":
            method,

        "run_name":
            run_name,

        "prediction_object":
            r["prediction_object"],

        "eval_split":
            params.get(
                "eval_split"
            ),

        "groups":
            params.get(
                "groups"
            ),

        "additional_instructions":
            params.get(
                "additional_instructions"
            ),
    })


manifest = pd.DataFrame(
    manifest_rows
)

print()
print(
    "manifest columns:",
    manifest.columns.tolist()
)

print(
    "MMLU runs:",
    len(manifest)
)

print(
    "Unique models:",
    manifest["model"].nunique()
)

print(
    "Unique subjects:",
    manifest["subject"].nunique()
)


# ------------------------------------------------------------
# 3. standard MMLU runだけ残す
# ------------------------------------------------------------

standard = manifest.copy()

standard = standard[
    standard["method"]
    == "multiple_choice_joint"
].copy()

standard = standard[
    standard[
        "additional_instructions"
    ].isna()
].copy()

standard = standard[
    standard["eval_split"].isna()
    |
    (
        standard["eval_split"]
        == "test"
    )
].copy()


# ------------------------------------------------------------
# 4. versionを追加
# ------------------------------------------------------------

standard["_version"] = (
    standard["suite"]
    .str.removeprefix("v")
    .map(Version)
)


# ------------------------------------------------------------
# 5. 同じ model × subject の場合は
#    新しいsuiteを優先
# ------------------------------------------------------------

standard = (
    standard
    .sort_values(
        [
            "model",
            "subject",
            "_version",
            "run_name",
        ],
        ascending=[
            True,
            True,
            False,
            True,
        ],
    )
)


selected = (
    standard
    .drop_duplicates(
        [
            "model",
            "subject",
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 6. 確認
# ------------------------------------------------------------

print()
print(
    "Final model-subject pairs:",
    len(selected)
)

print(
    "Final unique models:",
    selected["model"].nunique()
)

print(
    "Final unique subjects:",
    selected["subject"].nunique()
)


model_subject_counts = (
    selected
    .groupby("model")[
        "subject"
    ]
    .nunique()
    .sort_values(
        ascending=False
    )
)

print(
    "Models with all 57 subjects:",
    (
        model_subject_counts == 57
    ).sum()
)

display(
    model_subject_counts
)

prediction_manifest columns: ['suite', 'run_name', 'prediction_object']
display_predictions files: 4574

manifest columns: ['suite', 'subject', 'model', 'method', 'run_name', 'prediction_object', 'eval_split', 'groups', 'additional_instructions']
MMLU runs: 4574
Unique models: 81
Unique subjects: 57

Final model-subject pairs: 4559
Final unique models: 80
Final unique subjects: 57
Models with all 57 subjects: 79


model
01-ai_yi-34b                        57
01-ai_yi-6b                         57
mistralai_open-mistral-nemo-2407    57
mistralai_mixtral-8x7b-32kseqlen    57
mistralai_mixtral-8x22b             57
                                    ..
google_gemini-1.5-flash-001         57
google_gemini-1.0-pro-001           57
deepseek-ai_deepseek-v3             57
writer_palmyra-x-v3                 57
ai21_jamba-1.5-mini                 56
Name: subject, Length: 80, dtype: int64

In [31]:
all_subjects = set(MMLU_SUBJECTS)

for model, n_subjects in model_subject_counts.items():

    if n_subjects < 57:

        present = set(
            selected.loc[
                selected["model"] == model,
                "subject"
            ]
        )

        missing = sorted(
            all_subjects - present
        )

        print(
            model,
            "n_subjects =",
            n_subjects,
            "missing =",
            missing
        )

ai21_jamba-1.5-mini n_subjects = 56 missing = ['miscellaneous']


In [32]:
complete_models = (
    model_subject_counts[
        model_subject_counts == 57
    ]
    .index
    .tolist()
)

selected_complete = (
    selected[
        selected["model"].isin(
            complete_models
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Complete models:",
    selected_complete["model"].nunique()
)

print(
    "Model-subject runs:",
    len(selected_complete)
)

assert (
    selected_complete["model"].nunique()
    == 79
)

assert (
    len(selected_complete)
    == 79 * 57
)

Complete models: 79
Model-subject runs: 4503


In [28]:
model_subject_counts = (
    selected
    .groupby("model")[
        "subject"
    ]
    .nunique()
    .sort_values(
        ascending=False
    )
)


print(
    "Models with all 57 subjects:",
    (
        model_subject_counts == 57
    ).sum()
)

display(
    model_subject_counts
)

Models with all 57 subjects: 4


model
amazon_nova-lite-v1:0      57
amazon_nova-micro-v1:0     57
amazon_nova-pro-v1:0       57
deepseek-ai_deepseek-v3    57
Name: subject, dtype: int64

In [33]:
duplicates = (
    standard
    .groupby(
        ["model", "subject"]
    )
    .size()
    .reset_index(name="n_runs")
)

display(
    duplicates[
        duplicates["n_runs"] > 1
    ]
)

print(
    "Duplicate model-subject pairs:",
    (
        duplicates["n_runs"] > 1
    ).sum()
)

,model,subject,n_runs


Duplicate model-subject pairs: 0


In [34]:
standard["run_length"] = (
    standard["run_name"]
    .str.len()
)

selected = (
    standard
    .sort_values(
        [
            "model",
            "subject",
            "run_length",
            "run_name",
        ]
    )
    .drop_duplicates(
        ["model", "subject"],
        keep="first",
    )
    .reset_index(drop=True)
)


selected.to_csv(
    BASE_DIR
    / f"HELM_MMLU_{HELM_SUITE}_run_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)


print(
    "Selected model-subject runs:",
    len(selected)
)

print(
    "Models:",
    selected["model"].nunique()
)

print(
    "Subjects:",
    selected["subject"].nunique()
)

Selected model-subject runs: 4559
Models: 80
Subjects: 57


In [35]:
model_subject_counts = (
    selected
    .groupby("model")["subject"]
    .nunique()
    .sort_values(
        ascending=False
    )
)

display(
    model_subject_counts
)

print(
    "Models with all 57 subjects:",
    (model_subject_counts == 57).sum()
)
##5まで

model
01-ai_yi-34b                        57
01-ai_yi-6b                         57
mistralai_open-mistral-nemo-2407    57
mistralai_mixtral-8x7b-32kseqlen    57
mistralai_mixtral-8x22b             57
                                    ..
google_gemini-1.5-flash-001         57
google_gemini-1.0-pro-001           57
deepseek-ai_deepseek-v3             57
writer_palmyra-x-v3                 57
ai21_jamba-1.5-mini                 56
Name: subject, Length: 80, dtype: int64

Models with all 57 subjects: 79


In [36]:
def public_gcs_url(
    object_name
):
    """
    GCS public object URL.
    """

    encoded = quote(
        object_name,
        safe="/",
    )

    return (
        "https://storage.googleapis.com/"
        f"{BUCKET}/{encoded}"
    )


def download_json_with_retry(
    object_name,
    max_attempts=5,
):

    url = public_gcs_url(
        object_name
    )

    last_error = None

    for attempt in range(
        max_attempts
    ):

        try:

            r = session.get(
                url,
                timeout=60,
            )

            # 404等はretryしない
            if r.status_code == 404:
                raise FileNotFoundError(
                    object_name
                )

            r.raise_for_status()

            return r.json()

        except FileNotFoundError:
            raise

        except Exception as e:

            last_error = e

            if (
                attempt
                == max_attempts - 1
            ):
                raise

            wait = (
                2 ** attempt
                + random.random()
            )

            time.sleep(wait)

    raise last_error

In [37]:
def extract_helm_correct(stats):
    """
    HELMのinstance-level accuracyを0/1として取得。
    """

    if not isinstance(stats, dict):
        return np.nan

    # MMLUでは通常 exact_match
    candidates = [
        "exact_match",
        "quasi_exact_match",
    ]

    for key in candidates:

        if key not in stats:
            continue

        try:
            x = float(
                stats[key]
            )
        except Exception:
            continue

        if np.isfinite(x):
            return int(
                x >= 0.5
            )

    return np.nan

In [38]:
def process_helm_run(row):

    model = row["model"]
    subject = row["subject"]
    object_name = (
        row["prediction_object"]
    )

    try:

        data = (
            download_json_with_retry(
                object_name
            )
        )

        records = []

        for x in data:

            instance_id = x.get(
                "instance_id"
            )

            trial = x.get(
                "train_trial_index",
                0
            )

            predicted_text = x.get(
                "predicted_text"
            )

            stats = x.get(
                "stats",
                {}
            )

            correct = (
                extract_helm_correct(
                    stats
                )
            )

            records.append({
                "model": model,
                "subject": subject,
                "instance_id":
                    instance_id,
                "train_trial_index":
                    trial,
                "predicted_text":
                    predicted_text,
                "correct":
                    correct,
                "run_name":
                    row["run_name"],
                "prediction_object":
                    object_name,
            })

        df = pd.DataFrame(
            records
        )

        return {
            "status": "success",
            "model": model,
            "subject": subject,
            "data": df,
            "error": None,
        }

    except Exception as e:

        return {
            "status": "error",
            "model": model,
            "subject": subject,
            "data": None,
            "error": repr(e),
        }

In [39]:
results = []

with ThreadPoolExecutor(
    max_workers=N_WORKERS
) as executor:

    futures = {
        executor.submit(
            process_helm_run,
            row,
        ): (
            row["model"],
            row["subject"],
        )

        for _, row
        in selected_complete.iterrows()
    }

    n_total = len(futures)

    for i, future in enumerate(
        as_completed(futures),
        start=1,
    ):

        result = future.result()

        results.append(
            result
        )

        print(
            f"[{i}/{n_total}] "
            f"{result['model']} | "
            f"{result['subject']} | "
            f"{result['status']}"
        )

[1/4503] 01-ai_yi-34b | anatomy | success
[2/4503] 01-ai_yi-34b | abstract_algebra | success
[3/4503] 01-ai_yi-34b | clinical_knowledge | success
[4/4503] 01-ai_yi-34b | college_biology | success
[5/4503] 01-ai_yi-34b | astronomy | success
[6/4503] 01-ai_yi-34b | college_computer_science | success
[7/4503] 01-ai_yi-34b | college_chemistry | success
[8/4503] 01-ai_yi-34b | business_ethics | success
[9/4503] 01-ai_yi-34b | college_mathematics | success
[10/4503] 01-ai_yi-34b | college_physics | success
[11/4503] 01-ai_yi-34b | college_medicine | success
[12/4503] 01-ai_yi-34b | conceptual_physics | success
[13/4503] 01-ai_yi-34b | econometrics | success
[14/4503] 01-ai_yi-34b | computer_security | success
[15/4503] 01-ai_yi-34b | electrical_engineering | success
[16/4503] 01-ai_yi-34b | elementary_mathematics | success
[17/4503] 01-ai_yi-34b | formal_logic | success
[18/4503] 01-ai_yi-34b | global_facts | success
[19/4503] 01-ai_yi-34b | high_school_biology | success
[20/4503] 01-ai_yi-3

In [40]:
long_list = [
    x["data"]
    for x in results
    if (
        x["status"] == "success"
        and
        x["data"] is not None
    )
]


raw_long = pd.concat(
    long_list,
    ignore_index=True,
)


# 念のため trial=0 に限定
raw_long = raw_long[
    (
        raw_long[
            "train_trial_index"
        ].isna()
    )
    |
    (
        raw_long[
            "train_trial_index"
        ] == 0
    )
].copy()


# subjectを57分野順に
subject_order = {
    s: i
    for i, s
    in enumerate(MMLU_SUBJECTS)
}

raw_long["_subject_order"] = (
    raw_long["subject"]
    .map(subject_order)
)

raw_long = (
    raw_long
    .sort_values(
        [
            "model",
            "_subject_order",
            "instance_id",
        ]
    )
    .drop(
        columns="_subject_order"
    )
    .reset_index(drop=True)
)


raw_long.to_csv(
    BASE_DIR
    / f"HELM_MMLU_{HELM_SUITE}_raw_responses.csv",
    index=False,
    encoding="utf-8-sig",
)


print(
    raw_long.shape
)

display(
    raw_long.head(20)
)

(1109318, 8)


,model,subject,instance_id,train_trial_index,predicted_text,correct,run_name,prediction_object
0,01-ai_yi-34b,abstract_algebra,id100,0,A,0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
1,01-ai_yi-34b,abstract_algebra,id101,0,A,0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
2,01-ai_yi-34b,abstract_algebra,id102,0,A,1,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
3,01-ai_yi-34b,abstract_algebra,id103,0,A,1,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
4,01-ai_yi-34b,abstract_algebra,id104,0,C,0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
5,01-ai_yi-34b,abstract_algebra,id105,0,A,0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
6,01-ai_yi-34b,abstract_algebra,id106,0,A,0,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
7,01-ai_yi-34b,abstract_algebra,id107,0,B,1,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
8,01-ai_yi-34b,abstract_algebra,id108,0,B,1,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...
9,01-ai_yi-34b,abstract_algebra,id109,0,A,1,"mmlu:subject=abstract_algebra,method=multiple_...",mmlu/benchmark_output/runs/v1.0.0/mmlu:subject...


In [41]:
diagnostics = (
    raw_long
    .groupby(
        ["model", "subject"]
    )
    .agg(
        n_rows=(
            "instance_id",
            "size"
        ),
        n_unique_items=(
            "instance_id",
            "nunique"
        ),
        n_valid_correct=(
            "correct",
            lambda x:
                x.notna().sum()
        ),
        n_missing_correct=(
            "correct",
            lambda x:
                x.isna().sum()
        ),
        accuracy=(
            "correct",
            "mean"
        ),
    )
    .reset_index()
)


diagnostics.to_csv(
    BASE_DIR
    / f"HELM_MMLU_{HELM_SUITE}_diagnostics.csv",
    index=False,
    encoding="utf-8-sig",
)


display(
    diagnostics.sort_values(
        "n_missing_correct",
        ascending=False
    )
)

,model,subject,n_rows,n_unique_items,n_valid_correct,n_missing_correct,accuracy
0,01-ai_yi-34b,abstract_algebra,100,100,100,0,0.400000
3000,mistralai_mistral-large-2402,logical_fallacies,163,163,163,0,0.809816
3006,mistralai_mistral-large-2402,moral_disputes,346,346,346,0,0.829480
3005,mistralai_mistral-large-2402,miscellaneous,783,783,783,0,0.900383
3004,mistralai_mistral-large-2402,medical_genetics,100,100,100,0,0.740000
...,...,...,...,...,...,...,...
1507,google_gemini-1.5-flash-preview-0514,high_school_mathematics,270,270,270,0,0.500000
1508,google_gemini-1.5-flash-preview-0514,high_school_microeconomics,238,238,238,0,0.899160
1509,google_gemini-1.5-flash-preview-0514,high_school_physics,151,151,151,0,0.682119
1510,google_gemini-1.5-flash-preview-0514,high_school_psychology,545,545,545,0,0.946789


In [42]:
model_diagnostics = (
    diagnostics
    .groupby("model")
    .agg(
        n_subjects=(
            "subject",
            "nunique"
        ),
        n_rows=(
            "n_rows",
            "sum"
        ),
        n_valid=(
            "n_valid_correct",
            "sum"
        ),
        n_missing=(
            "n_missing_correct",
            "sum"
        ),
    )
    .reset_index()
)


model_diagnostics[
    "complete_57subjects"
] = (
    model_diagnostics[
        "n_subjects"
    ] == 57
)


model_diagnostics.to_csv(
    BASE_DIR
    / f"HELM_MMLU_{HELM_SUITE}_model_diagnostics.csv",
    index=False,
    encoding="utf-8-sig",
)


display(
    model_diagnostics
    .sort_values(
        [
            "complete_57subjects",
            "n_valid",
        ],
        ascending=[
            True,
            True,
        ]
    )
)

,model,n_subjects,n_rows,n_valid,n_missing,complete_57subjects
0,01-ai_yi-34b,57,14042,14042,0,True
1,01-ai_yi-6b,57,14042,14042,0,True
2,01-ai_yi-large-preview,57,14042,14042,0,True
3,ai21_jamba-1.5-large,57,14042,14042,0,True
4,ai21_jamba-instruct,57,14042,14042,0,True
...,...,...,...,...,...,...
74,qwen_qwen2.5-7b-instruct-turbo,57,14042,14042,0,True
75,snowflake_snowflake-arctic-instruct,57,14042,14042,0,True
76,upstage_solar-pro-241126,57,14042,14042,0,True
77,writer_palmyra-x-004,57,14042,14042,0,True


In [43]:
def instance_sort_key(x):

    x = str(x)

    m = re.search(
        r"(\d+)$",
        x
    )

    if m:
        return (
            0,
            int(m.group(1))
        )

    return (
        1,
        x
    )


subject_matrix_summary = []


for subject in MMLU_SUBJECTS:

    sub = raw_long[
        raw_long["subject"]
        == subject
    ].copy()

    if len(sub) == 0:

        print(
            "NO DATA:",
            subject
        )

        continue

    # 同一 model × item が重複していないか
    dup = (
        sub.duplicated(
            [
                "model",
                "instance_id",
            ],
            keep=False,
        )
    )

    if dup.any():

        print(
            "WARNING duplicate:",
            subject,
            dup.sum()
        )

        sub = (
            sub
            .sort_values(
                "train_trial_index"
            )
            .drop_duplicates(
                [
                    "model",
                    "instance_id",
                ],
                keep="first",
            )
        )

    matrix = sub.pivot(
        index="model",
        columns="instance_id",
        values="correct",
    )

    ordered_cols = sorted(
        matrix.columns,
        key=instance_sort_key,
    )

    matrix = matrix[
        ordered_cols
    ]

    matrix = (
        matrix
        .reset_index()
    )

    out_path = (
        SUBJECT_MATRIX_DIR
        / (
            "HELM_MMLU_correct_"
            f"{subject}.csv"
        )
    )

    matrix.to_csv(
        out_path,
        index=False,
        encoding="utf-8-sig",
    )

    subject_matrix_summary.append({
        "subject":
            subject,
        "n_models":
            len(matrix),
        "n_items":
            len(ordered_cols),
        "n_missing":
            int(
                matrix[
                    ordered_cols
                ]
                .isna()
                .sum()
                .sum()
            ),
        "file":
            str(out_path),
    })


subject_matrix_summary = (
    pd.DataFrame(
        subject_matrix_summary
    )
)

display(
    subject_matrix_summary
)


subject_matrix_summary.to_csv(
    BASE_DIR
    / "HELM_MMLU_subject_matrix_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

,subject,n_models,n_items,n_missing,file
0,abstract_algebra,79,100,0,260905 HELM MMLU data\correct_matrices_by_subj...
1,anatomy,79,135,0,260905 HELM MMLU data\correct_matrices_by_subj...
2,astronomy,79,152,0,260905 HELM MMLU data\correct_matrices_by_subj...
3,business_ethics,79,100,0,260905 HELM MMLU data\correct_matrices_by_subj...
4,clinical_knowledge,79,265,0,260905 HELM MMLU data\correct_matrices_by_subj...
5,college_biology,79,144,0,260905 HELM MMLU data\correct_matrices_by_subj...
6,college_chemistry,79,100,0,260905 HELM MMLU data\correct_matrices_by_subj...
7,college_computer_science,79,100,0,260905 HELM MMLU data\correct_matrices_by_subj...
8,college_mathematics,79,100,0,260905 HELM MMLU data\correct_matrices_by_subj...
9,college_medicine,79,173,0,260905 HELM MMLU data\correct_matrices_by_subj...


In [44]:
full = raw_long.copy()

full["item_key"] = (
    full["subject"]
    + "__"
    + full["instance_id"].astype(str)
)


# 重複確認
dup = full.duplicated(
    [
        "model",
        "item_key",
    ],
    keep=False,
)

print(
    "Duplicate model-item rows:",
    dup.sum()
)


if dup.any():

    full = (
        full
        .sort_values(
            "train_trial_index"
        )
        .drop_duplicates(
            [
                "model",
                "item_key",
            ],
            keep="first",
        )
    )


binary_matrix = full.pivot(
    index="model",
    columns="item_key",
    values="correct",
)


# subject順 → instance番号順
def full_item_sort_key(
    item
):

    subject, instance = (
        item.split(
            "__",
            1
        )
    )

    s_order = subject_order.get(
        subject,
        999
    )

    return (
        s_order,
        instance_sort_key(
            instance
        ),
    )


ordered_cols = sorted(
    binary_matrix.columns,
    key=full_item_sort_key,
)

binary_matrix = binary_matrix[
    ordered_cols
]


binary_matrix = (
    binary_matrix
    .reset_index()
)


binary_matrix.to_csv(
    BASE_DIR
    / (
        f"HELM_MMLU_{HELM_SUITE}"
        "_correct_binary_matrix.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


print(
    "Binary matrix shape:",
    binary_matrix.shape
)

print(
    "Number of item columns:",
    len(ordered_cols)
)

print(
    "Total NaNs:",
    binary_matrix[
        ordered_cols
    ].isna().sum().sum()
)

Duplicate model-item rows: 0
Binary matrix shape: (79, 14043)
Number of item columns: 14042
Total NaNs: 0


In [45]:
from pathlib import Path
from packaging.version import Version

import pandas as pd
import numpy as np
import json
import re
import unicodedata


# ============================================================
# instances.json 保存先
# ============================================================

INSTANCE_DIR = (
    BASE_DIR
    / "instances_by_subject"
)

INSTANCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# selected_complete:
# 先ほど作った79モデル × 57subject のrun manifest
#
# 必要列:
#   suite
#   subject
#   model
#   prediction_object
# ============================================================

tmp = selected_complete.copy()

tmp["_version"] = (
    tmp["suite"]
    .str.removeprefix("v")
    .map(Version)
)


# subjectごとに最も新しいsuiteのrunを1つ取る
representative_runs = (
    tmp
    .sort_values(
        [
            "subject",
            "_version",
            "model",
        ],
        ascending=[
            True,
            False,
            True,
        ]
    )
    .drop_duplicates(
        "subject",
        keep="first"
    )
    .reset_index(drop=True)
)


# display_predictions.json と同じrun directoryに
# instances.json がある
representative_runs[
    "instances_object"
] = (
    representative_runs[
        "prediction_object"
    ]
    .str.rsplit(
        "/",
        n=1
    )
    .str[0]
    +
    "/instances.json"
)


print(
    "Representative subjects:",
    representative_runs[
        "subject"
    ].nunique()
)

assert (
    representative_runs[
        "subject"
    ].nunique()
    == 57
)


display(
    representative_runs[
        [
            "subject",
            "suite",
            "model",
            "instances_object",
        ]
    ]
)

Representative subjects: 57


,subject,suite,model,instances_object
0,abstract_algebra,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...
1,anatomy,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...
2,astronomy,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...
3,business_ethics,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...
4,clinical_knowledge,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...
5,college_biology,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...
6,college_chemistry,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...
7,college_computer_science,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...
8,college_mathematics,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...
9,college_medicine,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...


In [46]:
all_object_set = set(
    all_objects
)


representative_runs[
    "instances_exists"
] = (
    representative_runs[
        "instances_object"
    ].isin(
        all_object_set
    )
)


missing_instance_files = (
    representative_runs[
        ~representative_runs[
            "instances_exists"
        ]
    ]
)


print(
    "instances.json found:",
    representative_runs[
        "instances_exists"
    ].sum(),
    "/ 57"
)


if len(
    missing_instance_files
) > 0:

    display(
        missing_instance_files[
            [
                "subject",
                "suite",
                "model",
                "instances_object",
            ]
        ]
    )

instances.json found: 57 / 57


In [48]:
instance_download_results = []


for i, row in representative_runs.iterrows():

    subject = row["subject"]

    object_name = (
        row["instances_object"]
    )

    print(
        f"[{i+1}/57] {subject}"
    )

    try:

        data = (
            download_json_with_retry(
                object_name
            )
        )

        save_path = (
            INSTANCE_DIR
            / f"{subject}.json"
        )

        with open(
            save_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False,
                indent=2
            )

        instance_download_results.append({
            "subject":
                subject,
            "status":
                "success",
            "n_instances":
                len(data),
            "suite":
                row["suite"],
            "model":
                row["model"],
            "instances_object":
                object_name,
            "local_file":
                str(save_path),
            "error":
                "",
        })

    except Exception as e:

        instance_download_results.append({
            "subject":
                subject,
            "status":
                "error",
            "n_instances":
                0,
            "suite":
                row["suite"],
            "model":
                row["model"],
            "instances_object":
                object_name,
            "local_file":
                "",
            "error":
                repr(e),
        })


instance_download_summary = (
    pd.DataFrame(
        instance_download_results
    )
)


display(
    instance_download_summary
)


print(
    "Successful subjects:",
    (
        instance_download_summary[
            "status"
        ]
        ==
        "success"
    ).sum()
)

[1/57] abstract_algebra
[2/57] anatomy
[3/57] astronomy
[4/57] business_ethics
[5/57] clinical_knowledge
[6/57] college_biology
[7/57] college_chemistry
[8/57] college_computer_science
[9/57] college_mathematics
[10/57] college_medicine
[11/57] college_physics
[12/57] computer_security
[13/57] conceptual_physics
[14/57] econometrics
[15/57] electrical_engineering
[16/57] elementary_mathematics
[17/57] formal_logic
[18/57] global_facts
[19/57] high_school_biology
[20/57] high_school_chemistry
[21/57] high_school_computer_science
[22/57] high_school_european_history
[23/57] high_school_geography
[24/57] high_school_government_and_politics
[25/57] high_school_macroeconomics
[26/57] high_school_mathematics
[27/57] high_school_microeconomics
[28/57] high_school_physics
[29/57] high_school_psychology
[30/57] high_school_statistics
[31/57] high_school_us_history
[32/57] high_school_world_history
[33/57] human_aging
[34/57] human_sexuality
[35/57] international_law
[36/57] jurisprudence
[37/57

,subject,status,n_instances,suite,model,instances_object,local_file,error
0,abstract_algebra,success,100,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\abs...,
1,anatomy,success,135,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\ana...,
2,astronomy,success,152,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\ast...,
3,business_ethics,success,100,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\bus...,
4,clinical_knowledge,success,265,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\cli...,
5,college_biology,success,144,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\col...,
6,college_chemistry,success,100,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\col...,
7,college_computer_science,success,100,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\col...,
8,college_mathematics,success,100,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\col...,
9,college_medicine,success,173,v1.13.0,amazon_nova-lite-v1:0,mmlu/benchmark_output/runs/v1.13.0/mmlu:subjec...,260905 HELM MMLU data\instances_by_subject\col...,


Successful subjects: 57


In [49]:
CORRECT_TAG_CANDIDATES = {
    "correct",
    "CORRECT",
    "correct_answer",
}


def get_text_from_output(x):
    """
    HELM Outputなどからtextを取得。
    """

    if x is None:
        return None

    if isinstance(x, str):
        return x

    if isinstance(x, dict):

        if "text" in x:
            return x["text"]

        if "output" in x:
            return get_text_from_output(
                x["output"]
            )

    return None


def get_input_text(instance):
    """
    HELM Instance.inputからquestion textを取得。
    """

    x = instance.get(
        "input"
    )

    if isinstance(x, str):
        return x

    if isinstance(x, dict):

        if "text" in x:
            return x["text"]

        # 将来/別形式への最低限の対応
        if "multimedia_content" in x:

            mm = x[
                "multimedia_content"
            ]

            if isinstance(mm, dict):

                content = mm.get(
                    "content",
                    []
                )

                texts = []

                for c in content:

                    if (
                        isinstance(c, dict)
                        and
                        c.get("text")
                        is not None
                    ):
                        texts.append(
                            str(c["text"])
                        )

                if texts:
                    return "\n".join(
                        texts
                    )

    return None


def is_correct_reference(ref):
    """
    Referenceのtagsからcorrectか判定。
    """

    tags = ref.get(
        "tags",
        []
    )

    if tags is None:
        tags = []

    tags = {
        str(x)
        for x in tags
    }

    return bool(
        tags
        &
        CORRECT_TAG_CANDIDATES
    )


def parse_helm_instances(
    subject,
    data
):

    records = []

    for position, instance in enumerate(
        data
    ):

        instance_id = instance.get(
            "id",
            instance.get(
                "instance_id",
                None
            )
        )

        question = (
            get_input_text(
                instance
            )
        )

        refs = instance.get(
            "references",
            []
        )

        choices = []
        correct_indices = []

        for j, ref in enumerate(refs):

            text = (
                get_text_from_output(
                    ref
                )
            )

            choices.append(
                text
            )

            if is_correct_reference(
                ref
            ):
                correct_indices.append(
                    j
                )

        if len(correct_indices) == 1:

            gold_index = (
                correct_indices[0]
            )

            gold = (
                "ABCD"[gold_index]
                if 0 <= gold_index < 4
                else None
            )

        else:

            gold_index = np.nan
            gold = None

        records.append({

            "subject":
                subject,

            "helm_position":
                position,

            "helm_instance_id":
                instance_id,

            "question":
                question,

            "choice_A":
                choices[0]
                if len(choices) > 0
                else None,

            "choice_B":
                choices[1]
                if len(choices) > 1
                else None,

            "choice_C":
                choices[2]
                if len(choices) > 2
                else None,

            "choice_D":
                choices[3]
                if len(choices) > 3
                else None,

            "n_choices":
                len(choices),

            "gold_index":
                gold_index,

            "gold":
                gold,

            "split":
                instance.get(
                    "split"
                ),
        })

    return pd.DataFrame(
        records
    )

In [50]:
helm_instance_tables = []


for subject in MMLU_SUBJECTS:

    path = (
        INSTANCE_DIR
        / f"{subject}.json"
    )

    if not path.exists():

        print(
            "Missing local file:",
            subject
        )

        continue

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    df = parse_helm_instances(
        subject,
        data
    )

    helm_instance_tables.append(
        df
    )


HELM_INSTANCES = pd.concat(
    helm_instance_tables,
    ignore_index=True
)


print(
    "HELM instance rows:",
    len(HELM_INSTANCES)
)

print(
    "Subjects:",
    HELM_INSTANCES[
        "subject"
    ].nunique()
)


display(
    HELM_INSTANCES.head()
)

HELM instance rows: 14042
Subjects: 57


,subject,helm_position,helm_instance_id,question,choice_A,choice_B,choice_C,choice_D,n_choices,gold_index,gold,split
0,abstract_algebra,0,id16,Find the degree for the given field extension ...,0,4,2,6,4,1.0,B,test
1,abstract_algebra,1,id17,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",8,2,24,120,4,2.0,C,test
2,abstract_algebra,2,id18,Find all zeros in the indicated finite field o...,0,1,"0,1","0,4",4,3.0,D,test
3,abstract_algebra,3,id19,Statement 1 | A factor group of a non-Abelian ...,"True, True","False, False","True, False","False, True",4,1.0,B,test
4,abstract_algebra,4,id20,Find the product of the given polynomials in t...,2x^2 + 5,6x^2 + 4x + 6,0,x^2 + 1,4,1.0,B,test


In [51]:
print(
    HELM_INSTANCES[
        "split"
    ].value_counts(
        dropna=False
    )
)

split
test    14042
Name: count, dtype: int64


In [52]:
HELM_TEST = (
    HELM_INSTANCES[
        HELM_INSTANCES[
            "split"
        ].astype(str)
        .str.lower()
        .eq("test")
    ]
    .copy()
)


print(
    "HELM test instances:",
    len(HELM_TEST)
)

HELM test instances: 14042


In [53]:
from datasets import load_dataset


canonical_rows = []


for subject in MMLU_SUBJECTS:

    print(
        "Loading canonical MMLU:",
        subject
    )

    ds = load_dataset(
        "cais/mmlu",
        subject,
        split="test"
    )

    for item_index, item in enumerate(
        ds
    ):

        choices = list(
            item["choices"]
        )

        gold_index = int(
            item["answer"]
        )

        canonical_rows.append({

            "subject":
                subject,

            "item_index":
                item_index,

            # Open LLM側と同じID
            "item_id":
                f"{subject}__{item_index:04d}",

            "question":
                item["question"],

            "choice_A":
                choices[0],

            "choice_B":
                choices[1],

            "choice_C":
                choices[2],

            "choice_D":
                choices[3],

            "gold_index":
                gold_index,

            "gold":
                "ABCD"[gold_index],
        })


MMLU_CANONICAL = pd.DataFrame(
    canonical_rows
)


print(
    len(MMLU_CANONICAL)
)

assert (
    len(MMLU_CANONICAL)
    == 14042
)

assert (
    MMLU_CANONICAL[
        "subject"
    ].nunique()
    == 57
)

Loading canonical MMLU: abstract_algebra


Loading canonical MMLU: anatomy
Loading canonical MMLU: astronomy
Loading canonical MMLU: business_ethics
Loading canonical MMLU: clinical_knowledge
Loading canonical MMLU: college_biology
Loading canonical MMLU: college_chemistry
Loading canonical MMLU: college_computer_science
Loading canonical MMLU: college_mathematics
Loading canonical MMLU: college_medicine
Loading canonical MMLU: college_physics
Loading canonical MMLU: computer_security
Loading canonical MMLU: conceptual_physics
Loading canonical MMLU: econometrics
Loading canonical MMLU: electrical_engineering
Loading canonical MMLU: elementary_mathematics
Loading canonical MMLU: formal_logic
Loading canonical MMLU: global_facts
Loading canonical MMLU: high_school_biology
Loading canonical MMLU: high_school_chemistry
Loading canonical MMLU: high_school_computer_science
Loading canonical MMLU: high_school_european_history
Loading canonical MMLU: high_school_geography
Loading canonical MMLU: high_school_government_and_politics
Loa

In [55]:
MMLU_CANONICAL["question"] = (
    MMLU_CANONICAL["question"]
    .astype(str)
    .str.strip()
)

In [56]:
OPENLLM_MASTER_PATH = (
    Path("260829 MMLU data")
    / "MMLU_master_14042_items.csv"
)


openllm_master = pd.read_csv(
    OPENLLM_MASTER_PATH,
    encoding="utf-8-sig"
)


check = (
    openllm_master[
        [
            "item_id",
            "subject",
            "item_index",
            "question",
            "gold",
        ]
    ]
    .merge(
        MMLU_CANONICAL[
            [
                "item_id",
                "subject",
                "item_index",
                "question",
                "gold",
            ]
        ],
        on="item_id",
        suffixes=(
            "_old",
            "_new"
        ),
        how="outer",
        indicator=True
    )
)


print(
    check["_merge"]
    .value_counts()
)


assert (
    check["_merge"]
    ==
    "both"
).all()


assert (
    check["subject_old"]
    ==
    check["subject_new"]
).all()


assert (
    check["item_index_old"]
    ==
    check["item_index_new"]
).all()


assert (
    check["question_old"]
    ==
    check["question_new"]
).all()


assert (
    check["gold_old"]
    ==
    check["gold_new"]
).all()


print(
    "OpenLLM master and canonical MMLU: EXACT MATCH"
)

_merge
both          14042
left_only         0
right_only        0
Name: count, dtype: int64
OpenLLM master and canonical MMLU: EXACT MATCH


In [57]:
def normalize_mmlu_text(x):

    if x is None:
        return ""

    if (
        isinstance(x, float)
        and np.isnan(x)
    ):
        return ""

    x = unicodedata.normalize(
        "NFKC",
        str(x)
    )

    x = x.replace(
        "\r\n",
        "\n"
    )

    x = x.replace(
        "\r",
        "\n"
    )

    # 空白差を吸収
    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x.strip()


KEY_COLUMNS = [
    "question",
    "choice_A",
    "choice_B",
    "choice_C",
    "choice_D",
]


def make_match_key(row):

    components = [
        normalize_mmlu_text(
            row["subject"]
        )
    ]

    for c in KEY_COLUMNS:

        components.append(
            normalize_mmlu_text(
                row[c]
            )
        )

    components.append(
        normalize_mmlu_text(
            row["gold"]
        )
    )

    return "\u241f".join(
        components
    )


MMLU_CANONICAL[
    "_match_key"
] = (
    MMLU_CANONICAL.apply(
        make_match_key,
        axis=1
    )
)


HELM_TEST[
    "_match_key"
] = (
    HELM_TEST.apply(
        make_match_key,
        axis=1
    )
)

In [58]:
# canonical側のkey重複を確認

canonical_duplicates = (
    MMLU_CANONICAL[
        "_match_key"
    ]
    .duplicated(
        keep=False
    )
)


print(
    "Canonical duplicate keys:",
    canonical_duplicates.sum()
)

Canonical duplicate keys: 54


In [59]:
MMLU_CANONICAL[
    "_occurrence"
] = (
    MMLU_CANONICAL
    .groupby(
        [
            "subject",
            "_match_key",
        ]
    )
    .cumcount()
)


HELM_TEST[
    "_occurrence"
] = (
    HELM_TEST
    .groupby(
        [
            "subject",
            "_match_key",
        ]
    )
    .cumcount()
)

In [60]:
HELM_ITEM_MAP = (
    HELM_TEST
    .merge(
        MMLU_CANONICAL[
            [
                "subject",
                "_match_key",
                "_occurrence",
                "item_id",
                "item_index",
            ]
        ],
        on=[
            "subject",
            "_match_key",
            "_occurrence",
        ],
        how="left",
        validate="one_to_one",
    )
)


HELM_ITEM_MAP[
    "matched"
] = (
    HELM_ITEM_MAP[
        "item_id"
    ].notna()
)


print(
    HELM_ITEM_MAP[
        "matched"
    ].value_counts()
)


print(
    "Matched:",
    HELM_ITEM_MAP[
        "matched"
    ].sum(),
    "/",
    len(HELM_ITEM_MAP)
)

matched
True     14038
False        4
Name: count, dtype: int64
Matched: 14038 / 14042


In [61]:
unmatched = (
    HELM_ITEM_MAP[
        ~HELM_ITEM_MAP[
            "matched"
        ]
    ]
    .copy()
)


print(
    "Unmatched:",
    len(unmatched)
)


if len(unmatched) > 0:

    display(
        unmatched[
            [
                "subject",
                "helm_instance_id",
                "question",
                "choice_A",
                "choice_B",
                "choice_C",
                "choice_D",
                "gold",
            ]
        ].head(50)
    )

Unmatched: 4


,subject,helm_instance_id,question,choice_A,choice_B,choice_C,choice_D,gold
2178,elementary_mathematics,id159,"Subtract. 2,396 – 1,709",687,687,"1,493","1,695",None
4021,high_school_macroeconomics,id231,A stronger stock market is likely to cause whi...,Increase Increase,No change No change,Increase No change,Increase Increase,None
6494,international_law,id69,How are the members of the arbitral tribunal a...,All the members of the arbitral tribunal are a...,All the members of the arbitral tribunal are a...,All the members of the arbitral tribunal are a...,All the members of the arbitral tribunal are a...,None
13601,sociology,id224,Economic aid has largely failed to promote mod...,there are no clearly defined projects into whi...,the United Nations has refused to call on rich...,debt repayments with interest can be greater t...,debt repayments with interest can be greater t...,None


In [62]:
# ============================================================
# 10. unmatched項目を gold を使わずに再対応付け
#
# 方針:
# - subject
# - question
# - choice_A ～ choice_D
#
# が一致すれば同一MMLU itemとみなす
#
# ただし、この方法で救済されたitemは
# HELM側では正解定義が曖昧なので、
# 後で correct = NaN にする
# ============================================================


def make_match_key_without_gold(row):

    components = [
        normalize_mmlu_text(
            row["subject"]
        ),
        normalize_mmlu_text(
            row["question"]
        ),
        normalize_mmlu_text(
            row["choice_A"]
        ),
        normalize_mmlu_text(
            row["choice_B"]
        ),
        normalize_mmlu_text(
            row["choice_C"]
        ),
        normalize_mmlu_text(
            row["choice_D"]
        ),
    ]

    return "\u241f".join(
        components
    )


# ------------------------------------------------------------
# canonical側
# ------------------------------------------------------------

MMLU_CANONICAL[
    "_match_key_nogold"
] = (
    MMLU_CANONICAL
    .apply(
        make_match_key_without_gold,
        axis=1
    )
)


# 同一内容が複数ある可能性にも対応
MMLU_CANONICAL[
    "_occurrence_nogold"
] = (
    MMLU_CANONICAL
    .sort_values(
        [
            "subject",
            "item_index"
        ]
    )
    .groupby(
        [
            "subject",
            "_match_key_nogold"
        ]
    )
    .cumcount()
)


# ------------------------------------------------------------
# HELM側
# ------------------------------------------------------------

HELM_ITEM_MAP[
    "_match_key_nogold"
] = (
    HELM_ITEM_MAP
    .apply(
        make_match_key_without_gold,
        axis=1
    )
)


HELM_ITEM_MAP[
    "_occurrence_nogold"
] = (
    HELM_ITEM_MAP
    .sort_values(
        [
            "subject",
            "helm_position"
        ]
    )
    .groupby(
        [
            "subject",
            "_match_key_nogold"
        ]
    )
    .cumcount()
)


# ------------------------------------------------------------
# 元々matchedだったかを保存
# ------------------------------------------------------------

HELM_ITEM_MAP[
    "matched_before_fix"
] = (
    HELM_ITEM_MAP[
        "matched"
    ].astype(bool)
)

In [63]:
# ============================================================
# 11. unmatchedだけをgold抜きで再merge
# ============================================================

unmatched_for_fix = (
    HELM_ITEM_MAP[
        ~HELM_ITEM_MAP[
            "matched_before_fix"
        ]
    ][
        [
            "subject",
            "helm_instance_id",
            "_match_key_nogold",
            "_occurrence_nogold",
        ]
    ]
    .copy()
)


canonical_for_fix = (
    MMLU_CANONICAL[
        [
            "subject",
            "_match_key_nogold",
            "_occurrence_nogold",
            "item_id",
            "item_index",
            "gold",
            "gold_index",
        ]
    ]
    .rename(
        columns={
            "item_id":
                "item_id_fix",

            "item_index":
                "item_index_fix",

            "gold":
                "gold_fix",

            "gold_index":
                "gold_index_fix",
        }
    )
)


fix_map = (
    unmatched_for_fix
    .merge(
        canonical_for_fix,
        on=[
            "subject",
            "_match_key_nogold",
            "_occurrence_nogold",
        ],
        how="left",
        validate="one_to_one",
    )
)


display(
    fix_map
)


print(
    "Unmatched items before fix:",
    len(unmatched_for_fix)
)

print(
    "Successfully mapped without gold:",
    fix_map[
        "item_id_fix"
    ].notna().sum()
)

,subject,helm_instance_id,_match_key_nogold,_occurrence_nogold,item_id_fix,item_index_fix,gold_fix,gold_index_fix
0,elementary_mathematics,id159,"elementary_mathematics␟Subtract. 2,396 – 1,709...",0,elementary_mathematics__0113,113,A,0
1,high_school_macroeconomics,id231,high_school_macroeconomics␟A stronger stock ma...,0,high_school_macroeconomics__0183,183,D,3
2,international_law,id69,international_law␟How are the members of the a...,0,international_law__0051,51,A,0
3,sociology,id224,sociology␟Economic aid has largely failed to p...,0,sociology__0197,197,C,2


Unmatched items before fix: 4
Successfully mapped without gold: 4


In [64]:
# ============================================================
# 12. fix結果をHELM_ITEM_MAPへ戻す
# ============================================================

HELM_ITEM_MAP = (
    HELM_ITEM_MAP
    .merge(
        fix_map[
            [
                "subject",
                "helm_instance_id",
                "item_id_fix",
                "item_index_fix",
                "gold_fix",
                "gold_index_fix",
            ]
        ],
        on=[
            "subject",
            "helm_instance_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# item_idが元々NaNだった4件だけ埋める
HELM_ITEM_MAP[
    "item_id"
] = (
    HELM_ITEM_MAP[
        "item_id"
    ]
    .fillna(
        HELM_ITEM_MAP[
            "item_id_fix"
        ]
    )
)


HELM_ITEM_MAP[
    "item_index"
] = (
    pd.to_numeric(
        HELM_ITEM_MAP[
            "item_index"
        ],
        errors="coerce"
    )
    .fillna(
        pd.to_numeric(
            HELM_ITEM_MAP[
                "item_index_fix"
            ],
            errors="coerce"
        )
    )
)


# ============================================================
# HELMで正解定義が曖昧だったitemのflag
#
# = 元々gold込みmatchingでは対応できなかったが、
#   gold抜きならcanonical itemに対応できたもの
# ============================================================

HELM_ITEM_MAP[
    "helm_ambiguous_gold"
] = (
    (~HELM_ITEM_MAP[
        "matched_before_fix"
    ])
    &
    (
        HELM_ITEM_MAP[
            "item_id_fix"
        ].notna()
    )
)


# fix後のmatched
HELM_ITEM_MAP[
    "matched"
] = (
    HELM_ITEM_MAP[
        "item_id"
    ].notna()
)


print(
    "Matched after fix:",
    HELM_ITEM_MAP[
        "matched"
    ].sum(),
    "/",
    len(HELM_ITEM_MAP)
)

print(
    "Still unmatched:",
    (
        ~HELM_ITEM_MAP[
            "matched"
        ]
    ).sum()
)

print(
    "HELM ambiguous-gold items:",
    HELM_ITEM_MAP[
        "helm_ambiguous_gold"
    ].sum()
)

Matched after fix: 14042 / 14042
Still unmatched: 0
HELM ambiguous-gold items: 4


In [66]:
# ============================================================
# fix結果をHELM_ITEM_MAPへ戻す
# 何度実行しても壊れない版
# ============================================================

# ------------------------------------------------------------
# 0. 前回の途中実行で残ったrepair列を削除
# ------------------------------------------------------------

repair_cols_to_drop = [
    c for c in HELM_ITEM_MAP.columns
    if (
        c.startswith("_repair_")
        or c in [
            "item_id_fix",
            "item_id_fix_x",
            "item_id_fix_y",
            "item_index_fix",
            "item_index_fix_x",
            "item_index_fix_y",
            "gold_fix",
            "gold_fix_x",
            "gold_fix_y",
            "gold_index_fix",
            "gold_index_fix_x",
            "gold_index_fix_y",
        ]
    )
]

HELM_ITEM_MAP = HELM_ITEM_MAP.drop(
    columns=repair_cols_to_drop,
    errors="ignore"
)


# ------------------------------------------------------------
# 1. 元々gold込みでmatchしていたかを保存
#    既に存在する場合はそのまま使う
# ------------------------------------------------------------

if "matched_before_fix" not in HELM_ITEM_MAP.columns:

    HELM_ITEM_MAP[
        "matched_before_fix"
    ] = (
        HELM_ITEM_MAP[
            "item_id"
        ].notna()
    )


# ------------------------------------------------------------
# 2. fix_map側の列名を、
#    絶対に衝突しない名前に変更
# ------------------------------------------------------------

repair_map = (
    fix_map[
        [
            "subject",
            "helm_instance_id",
            "item_id_fix",
            "item_index_fix",
            "gold_fix",
            "gold_index_fix",
        ]
    ]
    .rename(
        columns={
            "item_id_fix":
                "_repair_item_id",

            "item_index_fix":
                "_repair_item_index",

            "gold_fix":
                "_repair_gold",

            "gold_index_fix":
                "_repair_gold_index",
        }
    )
    .copy()
)


# ------------------------------------------------------------
# 3. merge
# ------------------------------------------------------------

HELM_ITEM_MAP = (
    HELM_ITEM_MAP
    .merge(
        repair_map,
        on=[
            "subject",
            "helm_instance_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 4. 元々item_idがNaNだった4問だけrepair値で埋める
# ------------------------------------------------------------

HELM_ITEM_MAP[
    "item_id"
] = (
    HELM_ITEM_MAP[
        "item_id"
    ]
    .fillna(
        HELM_ITEM_MAP[
            "_repair_item_id"
        ]
    )
)


HELM_ITEM_MAP[
    "item_index"
] = (
    pd.to_numeric(
        HELM_ITEM_MAP[
            "item_index"
        ],
        errors="coerce"
    )
    .fillna(
        pd.to_numeric(
            HELM_ITEM_MAP[
                "_repair_item_index"
            ],
            errors="coerce"
        )
    )
)


# ------------------------------------------------------------
# 5. HELMで正解定義が曖昧だった4問をflag
#
# 元々matchしなかった
# ＋
# goldを無視すればcanonical itemに対応できた
# ------------------------------------------------------------

HELM_ITEM_MAP[
    "helm_ambiguous_gold"
] = (
    ~HELM_ITEM_MAP[
        "matched_before_fix"
    ].astype(bool)
    &
    HELM_ITEM_MAP[
        "_repair_item_id"
    ].notna()
)


# ------------------------------------------------------------
# 6. fix後のmatchedを更新
# ------------------------------------------------------------

HELM_ITEM_MAP[
    "matched"
] = (
    HELM_ITEM_MAP[
        "item_id"
    ].notna()
)


# ------------------------------------------------------------
# 7. 確認
# ------------------------------------------------------------

print(
    "Matched after fix:",
    HELM_ITEM_MAP[
        "matched"
    ].sum(),
    "/",
    len(HELM_ITEM_MAP)
)

print(
    "Still unmatched:",
    (
        ~HELM_ITEM_MAP[
            "matched"
        ]
    ).sum()
)

print(
    "HELM ambiguous-gold items:",
    HELM_ITEM_MAP[
        "helm_ambiguous_gold"
    ].sum()
)


display(
    HELM_ITEM_MAP[
        HELM_ITEM_MAP[
            "helm_ambiguous_gold"
        ]
    ][
        [
            "subject",
            "helm_instance_id",
            "item_id",
            "item_index",
            "question",
            "choice_A",
            "choice_B",
            "choice_C",
            "choice_D",
            "_repair_gold",
            "_repair_gold_index",
        ]
    ]
)

Matched after fix: 14042 / 14042
Still unmatched: 0
HELM ambiguous-gold items: 4


,subject,helm_instance_id,item_id,item_index,question,choice_A,choice_B,choice_C,choice_D,_repair_gold,_repair_gold_index
2178,elementary_mathematics,id159,elementary_mathematics__0113,113.0,"Subtract. 2,396 – 1,709",687,687,"1,493","1,695",A,0.0
4021,high_school_macroeconomics,id231,high_school_macroeconomics__0183,183.0,A stronger stock market is likely to cause whi...,Increase Increase,No change No change,Increase No change,Increase Increase,D,3.0
6494,international_law,id69,international_law__0051,51.0,How are the members of the arbitral tribunal a...,All the members of the arbitral tribunal are a...,All the members of the arbitral tribunal are a...,All the members of the arbitral tribunal are a...,All the members of the arbitral tribunal are a...,A,0.0
13601,sociology,id224,sociology__0197,197.0,Economic aid has largely failed to promote mod...,there are no clearly defined projects into whi...,the United Nations has refused to call on rich...,debt repayments with interest can be greater t...,debt repayments with interest can be greater t...,C,2.0


In [67]:
# ============================================================
# canonical gold / gold_indexをitem_idから付与
# ============================================================

canonical_gold = (
    MMLU_CANONICAL[
        [
            "item_id",
            "gold",
            "gold_index",
        ]
    ]
    .rename(
        columns={
            "gold":
                "canonical_gold",

            "gold_index":
                "canonical_gold_index",
        }
    )
)


# 再実行に耐えるよう既存列を一旦削除
HELM_ITEM_MAP = (
    HELM_ITEM_MAP
    .drop(
        columns=[
            "canonical_gold",
            "canonical_gold_index",
        ],
        errors="ignore"
    )
)


HELM_ITEM_MAP = (
    HELM_ITEM_MAP
    .merge(
        canonical_gold,
        on="item_id",
        how="left",
        validate="many_to_one",
    )
)


assert (
    HELM_ITEM_MAP[
        "item_id"
    ].notna().all()
)

assert (
    HELM_ITEM_MAP[
        "canonical_gold"
    ].notna().all()
)


print(
    "All 14,042 HELM instances mapped "
    "to canonical MMLU items."
)

display(
    HELM_ITEM_MAP[
        HELM_ITEM_MAP[
            "helm_ambiguous_gold"
        ]
    ][
        [
            "subject",
            "helm_instance_id",
            "item_id",
            "question",
            "choice_A",
            "choice_B",
            "choice_C",
            "choice_D",
            "canonical_gold",
            "canonical_gold_index",
        ]
    ]
)

All 14,042 HELM instances mapped to canonical MMLU items.


,subject,helm_instance_id,item_id,question,choice_A,choice_B,choice_C,choice_D,canonical_gold,canonical_gold_index
2178,elementary_mathematics,id159,elementary_mathematics__0113,"Subtract. 2,396 – 1,709",687,687,"1,493","1,695",A,0
4021,high_school_macroeconomics,id231,high_school_macroeconomics__0183,A stronger stock market is likely to cause whi...,Increase Increase,No change No change,Increase No change,Increase Increase,D,3
6494,international_law,id69,international_law__0051,How are the members of the arbitral tribunal a...,All the members of the arbitral tribunal are a...,All the members of the arbitral tribunal are a...,All the members of the arbitral tribunal are a...,All the members of the arbitral tribunal are a...,A,0
13601,sociology,id224,sociology__0197,Economic aid has largely failed to promote mod...,there are no clearly defined projects into whi...,the United Nations has refused to call on rich...,debt repayments with interest can be greater t...,debt repayments with interest can be greater t...,C,2


In [68]:
# ============================================================
# 14. HELM → canonical MMLU item mapping保存
# ============================================================

HELM_ITEM_MAP_OUT = (
    HELM_ITEM_MAP[
        [
            "subject",
            "helm_position",
            "helm_instance_id",
            "item_id",
            "item_index",
            "question",
            "choice_A",
            "choice_B",
            "choice_C",
            "choice_D",
            "canonical_gold",
            "canonical_gold_index",
            "helm_ambiguous_gold",
            "matched",
        ]
    ]
    .rename(
        columns={
            "canonical_gold":
                "gold",

            "canonical_gold_index":
                "gold_index",
        }
    )
    .sort_values(
        [
            "subject",
            "item_index",
        ]
    )
    .reset_index(
        drop=True
    )
)


HELM_ITEM_MAP_OUT.to_csv(
    BASE_DIR
    / "HELM_to_OpenLLM_MMLU_item_mapping.csv",
    index=False,
    encoding="utf-8-sig",
)


print(
    "mapping rows:",
    len(HELM_ITEM_MAP_OUT)
)

print(
    "ambiguous HELM items:",
    HELM_ITEM_MAP_OUT[
        "helm_ambiguous_gold"
    ].sum()
)

mapping rows: 14042
ambiguous HELM items: 4


In [69]:
# ============================================================
# 15. raw HELM responsesにcanonical item_idを付与
# ============================================================

mapping_for_responses = (
    HELM_ITEM_MAP_OUT[
        [
            "subject",
            "helm_instance_id",
            "item_id",
            "item_index",
            "helm_ambiguous_gold",
        ]
    ]
    .rename(
        columns={
            "helm_instance_id":
                "instance_id"
        }
    )
)


raw_long_mapped = (
    raw_long
    .merge(
        mapping_for_responses,
        on=[
            "subject",
            "instance_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


print(
    "HELM response rows:",
    len(raw_long_mapped)
)

print(
    "Unmapped response rows:",
    raw_long_mapped[
        "item_id"
    ].isna().sum()
)

HELM response rows: 1109318
Unmapped response rows: 0


In [70]:
# ============================================================
# 16. HELMで正解定義が曖昧な4項目について
# 全モデルのcorrectをNaNにする
#
# item_id自体は維持する
# ============================================================

raw_long_mapped[
    "correct"
] = pd.to_numeric(
    raw_long_mapped[
        "correct"
    ],
    errors="coerce"
).astype(float)


n_before_nan = (
    raw_long_mapped[
        "correct"
    ].isna().sum()
)


ambiguous_response_mask = (
    raw_long_mapped[
        "helm_ambiguous_gold"
    ].fillna(False)
)


raw_long_mapped.loc[
    ambiguous_response_mask,
    "correct"
] = np.nan


n_after_nan = (
    raw_long_mapped[
        "correct"
    ].isna().sum()
)


print(
    "Ambiguous item-response cells set to NaN:",
    ambiguous_response_mask.sum()
)

print(
    "NaNs before:",
    n_before_nan
)

print(
    "NaNs after:",
    n_after_nan
)

print(
    "New NaNs added:",
    n_after_nan
    -
    n_before_nan
)

Ambiguous item-response cells set to NaN: 316
NaNs before: 0
NaNs after: 316
New NaNs added: 316


In [71]:
# ============================================================
# 17. ambiguous 4 itemsの確認
# ============================================================

ambiguous_item_ids = (
    HELM_ITEM_MAP_OUT.loc[
        HELM_ITEM_MAP_OUT[
            "helm_ambiguous_gold"
        ],
        "item_id"
    ]
    .tolist()
)


print(
    "Ambiguous canonical item IDs:"
)

for x in ambiguous_item_ids:
    print(x)


ambiguous_check = (
    raw_long_mapped[
        raw_long_mapped[
            "item_id"
        ].isin(
            ambiguous_item_ids
        )
    ]
    .groupby("item_id")
    .agg(
        n_models=(
            "model",
            "nunique"
        ),
        n_rows=(
            "correct",
            "size"
        ),
        n_nonmissing_correct=(
            "correct",
            lambda x:
                x.notna().sum()
        ),
        n_nan_correct=(
            "correct",
            lambda x:
                x.isna().sum()
        ),
    )
)


display(
    ambiguous_check
)

Ambiguous canonical item IDs:
elementary_mathematics__0113
high_school_macroeconomics__0183
international_law__0051
sociology__0197


,n_models,n_rows,n_nonmissing_correct,n_nan_correct
item_id,,,,
elementary_mathematics__0113,79,79,0,79
high_school_macroeconomics__0183,79,79,0,79
international_law__0051,79,79,0,79
sociology__0197,79,79,0,79


In [74]:
# ============================================================
# HELM model name -> Open LLM Leaderboard canonical name
#
# 同一モデルとして確認できるものだけ明示的に対応付け
# ============================================================

HELM_TO_OPENLLM_MODEL = {

    "01-ai_yi-34b":
        "01-ai/Yi-34B",

    "01-ai_yi-6b":
        "01-ai/Yi-6B",

    "allenai_olmo-1.7-7b":
        "allenai/OLMo-1.7-7B-hf",

    "allenai_olmo-7b":
        "allenai/OLMo-7B-hf",

    "databricks_dbrx-instruct":
        "databricks/dbrx-instruct",

    "deepseek-ai_deepseek-llm-67b-chat":
        "deepseek-ai/deepseek-llm-67b-chat",

    "google_gemma-7b":
        "google/gemma-7b",

    "meta_llama-2-13b":
        "meta-llama/Llama-2-13b-hf",

    "meta_llama-2-70b":
        "meta-llama/Llama-2-70b-hf",

    "meta_llama-2-7b":
        "meta-llama/Llama-2-7b-hf",

    "meta_llama-3-70b":
        "meta-llama/Meta-Llama-3-70B",

    "meta_llama-3-8b":
        "meta-llama/Meta-Llama-3-8B",

    "microsoft_phi-2":
        "microsoft/phi-2",

    "microsoft_phi-3-medium-4k-instruct":
        "microsoft/Phi-3-medium-4k-instruct",

    "mistralai_mistral-7b-v0.1":
        "mistralai/Mistral-7B-v0.1",

    "mistralai_mixtral-8x22b":
        "mistralai/Mixtral-8x22B-v0.1",

    "mistralai_mixtral-8x7b-32kseqlen":
        "mistralai/Mixtral-8x7B-v0.1",

    "qwen_qwen1.5-110b-chat":
        "Qwen/Qwen1.5-110B-Chat",

    "qwen_qwen1.5-14b":
        "Qwen/Qwen1.5-14B",

    "qwen_qwen1.5-32b":
        "Qwen/Qwen1.5-32B",

    "qwen_qwen1.5-72b":
        "Qwen/Qwen1.5-72B",

    "qwen_qwen1.5-7b":
        "Qwen/Qwen1.5-7B",
}


# ============================================================
# original HELM nameを保存
# ============================================================

if "model_helm_original" not in raw_long_mapped.columns:

    raw_long_mapped[
        "model_helm_original"
    ] = raw_long_mapped[
        "model"
    ]


# ============================================================
# canonical名へ置換
#
# mappingにないHELM-onlyモデルは元の名前のまま
# ============================================================

raw_long_mapped[
    "model"
] = (
    raw_long_mapped[
        "model_helm_original"
    ]
    .replace(
        HELM_TO_OPENLLM_MODEL
    )
)


print(
    "HELM models:",
    raw_long_mapped[
        "model"
    ].nunique()
)

print(
    "Renamed overlapping models:",
    raw_long_mapped[
        "model_helm_original"
    ]
    .isin(
        HELM_TO_OPENLLM_MODEL
    )
    .groupby(
        raw_long_mapped[
            "model_helm_original"
        ]
    )
    .any()
    .sum()
)

HELM models: 79
Renamed overlapping models: 22


In [75]:
# ============================================================
# 18. HELM subject別 binary response matrix
#
# 値:
#   1.0 = correct
#   0.0 = incorrect
#   NaN = missing / ambiguous HELM item
# ============================================================

HELM_BINARY_DIR = (
    BASE_DIR
    / "binary_matrices_by_subject"
)

HELM_BINARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


matrix_summary = []


for subject in MMLU_SUBJECTS:

    canonical_items = (
        MMLU_CANONICAL[
            MMLU_CANONICAL[
                "subject"
            ]
            ==
            subject
        ]
        .sort_values(
            "item_index"
        )
        .copy()
    )


    ordered_item_ids = (
        canonical_items[
            "item_id"
        ]
        .astype(str)
        .tolist()
    )


    sub = (
        raw_long_mapped[
            raw_long_mapped[
                "subject"
            ]
            ==
            subject
        ]
        .copy()
    )


    # --------------------------------------------------------
    # model × item重複チェック
    # --------------------------------------------------------

    dup = sub.duplicated(
        [
            "model",
            "item_id",
        ],
        keep=False
    )


    if dup.any():

        print(
            "WARNING duplicate:",
            subject,
            int(
                dup.sum()
            )
        )

        sub = (
            sub
            .drop_duplicates(
                [
                    "model",
                    "item_id",
                ],
                keep="first"
            )
        )


    matrix = (
        sub
        .pivot(
            index="model",
            columns="item_id",
            values="correct",
        )
    )


    # canonical MMLUと同じ列順へ
    matrix = (
        matrix
        .reindex(
            columns=
                ordered_item_ids
        )
    )


    matrix = (
        matrix
        .reset_index()
    )


    out_path = (
        HELM_BINARY_DIR
        /
        f"HELM_MMLU_correct_{subject}.csv"
    )


    matrix.to_csv(
        out_path,
        index=False,
        encoding="utf-8-sig"
    )


    n_nan = int(
        matrix[
            ordered_item_ids
        ]
        .isna()
        .sum()
        .sum()
    )


    matrix_summary.append({
        "subject":
            subject,

        "n_models":
            len(matrix),

        "n_items":
            len(
                ordered_item_ids
            ),

        "n_nan":
            n_nan,

        "contains_ambiguous_item":
            any(
                x
                in
                ambiguous_item_ids
                for x
                in ordered_item_ids
            ),

        "file":
            str(out_path),
    })


matrix_summary_df = (
    pd.DataFrame(
        matrix_summary
    )
)


display(
    matrix_summary_df
)


matrix_summary_df.to_csv(
    BASE_DIR
    / "HELM_MMLU_binary_matrix_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

,subject,n_models,n_items,n_nan,contains_ambiguous_item,file
0,abstract_algebra,79,100,0,False,260905 HELM MMLU data\binary_matrices_by_subje...
1,anatomy,79,135,0,False,260905 HELM MMLU data\binary_matrices_by_subje...
2,astronomy,79,152,0,False,260905 HELM MMLU data\binary_matrices_by_subje...
3,business_ethics,79,100,0,False,260905 HELM MMLU data\binary_matrices_by_subje...
4,clinical_knowledge,79,265,0,False,260905 HELM MMLU data\binary_matrices_by_subje...
5,college_biology,79,144,0,False,260905 HELM MMLU data\binary_matrices_by_subje...
6,college_chemistry,79,100,0,False,260905 HELM MMLU data\binary_matrices_by_subje...
7,college_computer_science,79,100,0,False,260905 HELM MMLU data\binary_matrices_by_subje...
8,college_mathematics,79,100,0,False,260905 HELM MMLU data\binary_matrices_by_subje...
9,college_medicine,79,173,0,False,260905 HELM MMLU data\binary_matrices_by_subje...


In [76]:
# ============================================================
# 19. OpenLLM probability matrixとのitem列一致確認
# ============================================================

OPENLLM_MATRIX_DIR = (
    Path(
        "260829 MMLU data"
    )
    /
    "p_correct_matrices_by_subject"
)


validation_rows = []


for subject in MMLU_SUBJECTS:

    open_path = (
        OPENLLM_MATRIX_DIR
        /
        f"MMLU_p_correct_{subject}.csv"
    )

    helm_path = (
        HELM_BINARY_DIR
        /
        f"HELM_MMLU_correct_{subject}.csv"
    )


    open_cols = (
        pd.read_csv(
            open_path,
            nrows=0
        )
        .columns
        .tolist()
    )


    helm_cols = (
        pd.read_csv(
            helm_path,
            nrows=0
        )
        .columns
        .tolist()
    )


    open_items = (
        open_cols[1:]
    )

    helm_items = (
        helm_cols[1:]
    )


    validation_rows.append({

        "subject":
            subject,

        "n_openllm_items":
            len(open_items),

        "n_helm_items":
            len(helm_items),

        "same_columns":
            open_items
            ==
            helm_items,
    })


column_validation = (
    pd.DataFrame(
        validation_rows
    )
)


display(
    column_validation
)


print(
    "All 57 subjects identical:",
    column_validation[
        "same_columns"
    ].all()
)

,subject,n_openllm_items,n_helm_items,same_columns
0,abstract_algebra,100,100,True
1,anatomy,135,135,True
2,astronomy,152,152,True
3,business_ethics,100,100,True
4,clinical_knowledge,265,265,True
5,college_biology,144,144,True
6,college_chemistry,100,100,True
7,college_computer_science,100,100,True
8,college_mathematics,100,100,True
9,college_medicine,173,173,True


All 57 subjects identical: True
